# Track A 08: Complete Qwen2.5-3B Reference Experiment

This self-contained Colab notebook collects the complete registered Qwen evidence in `CSE407_qwen_track_a_reference_v1/`: 75 core energy rows, 20 sensitivity energy rows, 95 raw traces, 300 core hard-quality rows, and 60 sensitivity-quality rows.

Q1-S is parity-only. Q3-Ref and Q4-Ref use `turboquant==0.2.0` as a numerical reference path; it is not packed, fused, or physically 4-bit.


## Run Order and Immutable Boundary

Run sections in numeric order on a fresh Colab T4. The notebook may resume compatible completed blocks and quality shards, but rejects every mismatched schema, fingerprint, source hash, schedule, prompt, trace, or gate. Energy blocks are staged and promoted only as complete paired units. `analysis/final_integrity.json` is the final filesystem mutation.

Never read measurement rows from or write into accepted Llama directories or `CSE407_qwen_track_a_adapter_pilot_v1/`.


## 1. Install the Pinned Stack

Run once, restart the runtime, then continue with section 2.


In [1]:
%pip install -q "transformers==5.12.1" "gptqmodel==7.1.0" "turboquant==0.2.0"
%pip install -q "accelerate==1.13.0" "safetensors==0.7.0" "tokenizers==0.22.2" "huggingface-hub==1.5.0"
%pip install -q "numpy==2.2.6" "scipy==1.16.1" nvidia-ml-py pandas matplotlib ninja maturin


## 2. Runtime, Source, and Immutable Protocol Gates


In [2]:
import gc
import hashlib
import importlib.metadata
import inspect
import json
import math
import os
import platform
import random
import re
import shutil
import threading
import time
from collections.abc import Mapping
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pynvml
import scipy
import scipy.stats as stats
import torch
import torch.nn.functional as F
import transformers
import transformers.cache_utils as transformers_cache_utils
import transformers.masking_utils as transformers_masking_utils
import transformers.integrations.sdpa_attention as transformers_sdpa_attention
import turboquant
from huggingface_hub import HfApi
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer, DynamicCache, set_seed
from transformers.modeling_utils import ALL_ATTENTION_FUNCTIONS
from transformers.models.qwen2.modeling_qwen2 import Qwen2Attention, apply_rotary_pos_emb, eager_attention_forward, repeat_kv
from turboquant import TurboQuantCache, TurboQuantMSE

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
SEED = 407
SCHEMA_VERSION = "track-a-qwen-full-reference-v1-2026-08-04-1"
EXPECTED_FINGERPRINT = "861d89617e53034d"
ARTIFACT_NAME = "CSE407_qwen_track_a_reference_v1"
MODEL_FP16 = "Qwen/Qwen2.5-3B-Instruct"
MODEL_FP16_REVISION = "aa8e72537993ba99e69dfaafa59ed015b17504d1"
MODEL_AWQ = "Qwen/Qwen2.5-3B-Instruct-AWQ"
MODEL_AWQ_REVISION = "3559b226e8ce77211e2c1bd7ddfb7686fec4d6dd"
TRANSFORMERS_COMMIT = "ddb849abe009d1089e6c691bfc897f27211c663c"
SOURCE_HASHES = {
    "qwen2": "99fa98c5676604cf6ef505892b70fda5c1c4cd835971459f38d61090cccab1e4",
    "cache_utils": "2783b2702910dbb6e70aa05a093f71e1c51c110d7fa73c904fc38d12e5882545",
    "masking_utils": "aa525d800f824b25cbe9542ca7cc5b35e948e7cfa3178e2c0f2356a15ff32df2",
    "sdpa_attention": "87f933d1a2d8508df572da5c0748c6b24c22ff2b625796949957dcd86cc57564",
    "turboquant": "7358e2778c8de6974b0c7212ae8e3dbe00552a96b2e2df0aa42878f239a4c72a",
}
EXPECTED_SEMANTIC_SHA256 = "fffa82607af6e7d2c892d18d5bc790524570833472ad397b7ab2e54eb03005ef"
EXPECTED_PROMPT_SET_SHA256 = "115c1568fea7aeea038a46ce309d0c8bcaf1f5405544466ad6f379ee7e94ae3a"
EXPECTED_TOKENIZER_BACKEND_SHA256 = "e360e6f6266884e6667874f3a7d158ca20c4639e2e1dc80427e1edc2c69a38c7"
EXPECTED_CHAT_TEMPLATE_SHA256 = "cd8e9439f0570856fd70470bf8889ebd8b5d1107207f67a5efb46e342330527f"
EXPECTED_SPECIAL_TOKENS_SHA256 = "edcb2fc2acbbe77f858a9c4fe51295ffdb84711efba5703ec5906b3d67282569"
EXPECTED_ENERGY_PROMPT_HASHES = {
    "1024": "372707bd5b0af49a6ba6e7701df7fce5d5abb425fd2455e7960402cfabe7beff",
    "2048": "0c522a34a53d62ffb33827069f708e49128be93cff1f42a09813d7422deb282c",
    "4096": "39f4572a840a60eaf9040cb1a9aefe1b1212819ad5791a306ba39fa194de2f5b",
}
EXPECTED_DECODE_IDS_SHA256 = "b4d56dec09bd027278c2b39e7b397b9501328955aeef32e2155204f8d0920889"
EXPECTED_DECODE_TEXT_SHA256 = "37d78b47fba6700cdf7ccc6faf8dd2aa24fbd0124544d5b9604d02643fc768c4"
EXPECTED_SCHEDULE_SHA256 = "3784f684a6da44aed47dc32725ed1c9148c00a920292c4f5e1e4deb404aa1373"

CONTEXTS = (1024, 2048, 4096)
CONTEXT_TOKENS = QUALITY_CONTEXT = 4096
MEASURED_BLOCKS = 5
FIXED_DECODE_TOKENS = 64
QUALITY_MAX_NEW = 32
NUM_LAYERS, NUM_Q_HEADS, NUM_KV_HEADS, HEAD_DIM = 36, 16, 2, 128
HIDDEN_SIZE, INTERMEDIATE_SIZE, VOCAB_SIZE, MAX_POSITIONS = 2048, 11008, 151936, 32768
ROPE_THETA = 1_000_000.0
WINDOW, POOL_KERNEL, PYRAMID_BETA = 32, 5, 20
PARITY_ATOL = PARITY_RTOL = 5e-3
POLL_INTERVAL_S = 0.1
POWER_CAPS_W = (82.0, 70.0)
TQ_NMSE_MAX_BY_BITS = {2: 1.0, 3: 0.5, 4: 0.25}
CORE_VARIANTS = ("Q0", "Q1-W", "Q2", "Q3-Ref", "Q4-Ref")
AWQ_CORE_VARIANTS = ("Q1-W", "Q2", "Q3-Ref", "Q4-Ref")
SENSITIVITY_VARIANTS = ("Q2-B512", "Q2-B1536", "Q3-Ref-TQ2", "Q3-Ref-TQ3")
EXPECTED_COUNTS = {"core_energy": 75, "sensitivity_energy": 20, "traces": 95, "core_quality": 300, "sensitivity_quality": 60}
FINGERPRINT_VARIANT_SPECS = {
    "Q0": {"wrapper": False, "eviction": False, "budget": 1024, "tq": False, "bits": 4},
    "Q1-S": {"wrapper": False, "eviction": False, "budget": 1024, "tq": False, "bits": 4},
    "Q1-W": {"wrapper": True, "eviction": False, "budget": 1024, "tq": False, "bits": 4},
    "Q2": {"wrapper": True, "eviction": True, "budget": 1024, "tq": False, "bits": 4},
    "Q3-Ref": {"wrapper": True, "eviction": False, "budget": 1024, "tq": True, "bits": 4},
    "Q4-Ref": {"wrapper": True, "eviction": True, "budget": 1024, "tq": True, "bits": 4},
    "Q2-B512": {"wrapper": True, "eviction": True, "budget": 512, "tq": False, "bits": 4},
    "Q2-B1536": {"wrapper": True, "eviction": True, "budget": 1536, "tq": False, "bits": 4},
    "Q3-Ref-TQ2": {"wrapper": True, "eviction": False, "budget": 1024, "tq": True, "bits": 2},
    "Q3-Ref-TQ3": {"wrapper": True, "eviction": False, "budget": 1024, "tq": True, "bits": 3},
}
FORBIDDEN_ARTIFACT_NAMES = {
    "CSE407_llama_v5_final", "CSE407_llama_track_a_full_v1",
    "CSE407_llama_power_robustness_v1", "CSE407_qwen_track_a_adapter_pilot_v1",
    "CSE407_llama_track_a_pilot_discarded", "CSE407_llama_track_a_pilot_v2_discarded",
    "CSE407_llama_track_a_pilot_v3_discarded", "CSE407_llama_track_a_pilot_v4_discarded",
}

set_seed(SEED); random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
assert torch.cuda.is_available()
GPU = torch.cuda.get_device_properties(0)
assert "T4" in GPU.name
assert transformers.__version__ == "5.12.1"
assert importlib.metadata.version("gptqmodel") == "7.1.0"
assert importlib.metadata.version("turboquant") == "0.2.0"
assert importlib.metadata.version("tokenizers") == "0.22.2"
assert np.__version__ == "2.2.6" and scipy.__version__ == "1.16.1"
installed_source_hashes = {
    "qwen2": hashlib.sha256(Path(inspect.getsourcefile(Qwen2Attention)).read_bytes()).hexdigest(),
    "cache_utils": hashlib.sha256(Path(inspect.getsourcefile(transformers_cache_utils)).read_bytes()).hexdigest(),
    "masking_utils": hashlib.sha256(Path(inspect.getsourcefile(transformers_masking_utils)).read_bytes()).hexdigest(),
    "sdpa_attention": hashlib.sha256(Path(inspect.getsourcefile(transformers_sdpa_attention)).read_bytes()).hexdigest(),
    "turboquant": hashlib.sha256(b"".join(p.read_bytes() for p in sorted(Path(turboquant.__file__).parent.glob("*.py")))).hexdigest(),
}
assert installed_source_hashes == SOURCE_HASHES
forward_parameters = tuple(inspect.signature(Qwen2Attention.forward).parameters)
assert forward_parameters == ("self", "hidden_states", "position_embeddings", "attention_mask", "past_key_values", "kwargs")
assert "cache_position" not in forward_parameters
api = HfApi()
assert api.model_info(MODEL_FP16, revision=MODEL_FP16_REVISION).sha == MODEL_FP16_REVISION
assert api.model_info(MODEL_AWQ, revision=MODEL_AWQ_REVISION).sha == MODEL_AWQ_REVISION
pynvml.nvmlInit(); NVML_HANDLE = pynvml.nvmlDeviceGetHandleByIndex(0)
enforced_power_limit_w = pynvml.nvmlDeviceGetEnforcedPowerLimit(NVML_HANDLE) / 1000.0
assert np.isclose(enforced_power_limit_w, 70.0, atol=0.5), enforced_power_limit_w
SESSION_ID = time.strftime("%Y%m%dT%H%M%S") + "-" + os.urandom(4).hex()
print(GPU.name, "runtime/source gates passed; enforced limit", enforced_power_limit_w, "W")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Tesla T4 runtime/source gates passed; enforced limit 70.0 W


## 3. Embedded Frozen V4 Generator and Strict Scorer

This accepted semantic generator is embedded unchanged. Do not tune it.


In [3]:
GENERATOR_VERSION = "track-a-hard-retrieval-v4-controlled-pairs-draft"
MANIFEST_SCHEMA = "cse407-hard-retrieval-manifest-v1"
BASE_SEED = 407_100
DEPTHS = (0.1, 0.5, 0.9)
SCENARIOS_PER_DEPTH = 20
RECORD_COUNT = 32
FINAL_QUERY_WINDOW = 32
MIN_RECORD_GAP_TOKENS = 16

CODE_PATTERN = re.compile(
    r"(?<![A-Z0-9])([A-H]\d{3}-[K-NP-Q]\d{2})(?![A-Z0-9])",
    re.IGNORECASE,
)
PARTIAL_CODE_PATTERN = re.compile(
    r"(?<![A-Z0-9])(?:[A-H]\d{1,2}|[A-H]\d{3}(?:-(?:[K-NP-Q]?\d?)?)?)(?![A-Z0-9])",
    re.IGNORECASE,
)
SYSTEM_INSTRUCTION = (
    "Use only the document. Return exactly one activation code and no explanation."
)
PLACEHOLDER = "{HARD_RETRIEVAL_DOCUMENT}"
KEY_LABELS = (
    "AURORA", "BOREAL", "CEDAR", "DRAGON", "EMBER", "FALCON", "GARNET", "HARBOR",
    "INDIGO", "JASPER", "KESTREL", "LANTERN", "MARINER", "NEBULA", "ORBIT", "PRAIRIE",
    "QUARTZ", "RAVEN", "SUMMIT", "TEMPEST", "UMBRA", "VECTOR", "WILLOW", "XENON",
    "YARROW", "ZEPHYR", "ORBITAL", "MARINA", "CEDRIC", "FALCONE", "GARNET-X", "HARBOR-X",
)
FILLER_TOPICS = (
    "thermal scheduling", "memory allocation", "coastal monitoring",
    "warehouse routing", "battery maintenance", "traffic forecasting",
    "compiler testing", "water treatment", "satellite imaging",
    "clinical logistics", "network telemetry", "acoustic sensing",
    "grid balancing", "crop analysis", "robot calibration", "archive indexing",
)
FILLER_METHODS = (
    "compares recent measurements with a rolling baseline",
    "records each decision before the next review cycle",
    "uses independent checks to detect inconsistent observations",
    "separates routine maintenance from exceptional interventions",
    "prioritizes stable operation over short-lived peak performance",
    "tracks uncertainty alongside every summarized measurement",
    "repeats validation after any material configuration change",
    "keeps raw observations available for later auditing",
)
FILLER_OUTCOMES = (
    "The review found that seasonal variation explained most of the change.",
    "Operators retained the conservative threshold for the next evaluation.",
    "No single measurement was treated as sufficient evidence for deployment.",
    "The team documented both successful trials and inconclusive observations.",
    "A later audit confirmed the ordering but revised the uncertainty estimate.",
    "The report emphasized reproducibility and clearly separated derived values.",
    "Follow-up work focused on failure recovery rather than headline throughput.",
    "The final recommendation required another independent measurement period.",
)


def canonical_hash(value):
    payload = json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=True)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def token_ids_hash(input_ids):
    return hashlib.sha256(",".join(map(str, input_ids)).encode("ascii")).hexdigest()


def _make_unique_codes(rng, count):
    letters = "ABCDEFGH"
    suffixes = "KLMNPQ"
    codes = set()
    while len(codes) < count:
        codes.add(
            f"{rng.choice(letters)}{rng.randrange(100, 1000):03d}-"
            f"{rng.choice(suffixes)}{rng.randrange(10, 100):02d}"
        )
    result = sorted(codes)
    rng.shuffle(result)
    return result


def _make_similar_keys(rng, count):
    candidates = [f"NODE-{label}" for label in KEY_LABELS]
    if count > len(candidates):
        raise ValueError("Controlled-pair pilot supports at most 32 records")
    rng.shuffle(candidates)
    return candidates[:count]


def validate_semantic_scenario(scenario):
    records = scenario["records"]
    if not 32 <= len(records) <= 64:
        raise ValueError("Each scenario must contain 32 to 64 records")
    keys = [record["key"] for record in records]
    codes = [record["code"] for record in records]
    if len(set(keys)) != len(keys) or len(set(codes)) != len(codes):
        raise ValueError("Record keys and codes must be unique")
    if any(CODE_PATTERN.fullmatch(code) is None for code in codes):
        raise ValueError("Every record code must match the declared answer format")
    targets = [record for record in records if record["is_target"]]
    if len(targets) != 1:
        raise ValueError("Each scenario must have exactly one target record")
    target = targets[0]
    if target["key"] != scenario["target_key"] or target["code"] != scenario["target_code"]:
        raise ValueError("Target metadata does not match the target record")
    if scenario["depth"] not in DEPTHS:
        raise ValueError(f"Unsupported target depth: {scenario['depth']}")


def build_semantic_manifest(
    depths=DEPTHS,
    scenarios_per_depth=SCENARIOS_PER_DEPTH,
    record_count=RECORD_COUNT,
    base_seed=BASE_SEED,
):
    if not 32 <= record_count <= 64:
        raise ValueError("record_count must be between 32 and 64")
    scenarios = []
    for depth_index, depth in enumerate(depths):
        if depth not in DEPTHS:
            raise ValueError(f"Unsupported target depth: {depth}")
        for index in range(scenarios_per_depth):
            scenario_seed = base_seed + depth_index * 100_000 + index
            rng = random.Random(scenario_seed)
            keys = _make_similar_keys(rng, record_count)
            codes = _make_unique_codes(rng, record_count)
            target_index = rng.randrange(record_count)
            records = [
                {"key": key, "code": code, "is_target": position == target_index}
                for position, (key, code) in enumerate(zip(keys, codes))
            ]
            rng.shuffle(records)
            target = next(record for record in records if record["is_target"])
            scenario = {
                "scenario_id": f"d{int(depth * 100):02d}-s{index:02d}",
                "seed": scenario_seed,
                "depth": depth,
                "target_key": target["key"],
                "target_code": target["code"],
                "records": records,
            }
            validate_semantic_scenario(scenario)
            scenarios.append(scenario)
    manifest = {
        "manifest_schema": MANIFEST_SCHEMA,
        "generator_version": GENERATOR_VERSION,
        "status": "DRAFT_UNTIL_C1W_C2_PILOT_PASSES",
        "base_seed": base_seed,
        "depths": list(depths),
        "scenarios_per_depth": scenarios_per_depth,
        "record_count": record_count,
        "scenarios": scenarios,
    }
    manifest["semantic_sha256"] = canonical_hash(manifest)
    return manifest


def _make_filler_text(rng, minimum_paragraphs=512):
    paragraphs = []
    topic_order = list(FILLER_TOPICS)
    method_order = list(FILLER_METHODS)
    outcome_order = list(FILLER_OUTCOMES)
    for index in range(minimum_paragraphs):
        if index % len(topic_order) == 0:
            rng.shuffle(topic_order)
        if index % len(method_order) == 0:
            rng.shuffle(method_order)
        if index % len(outcome_order) == 0:
            rng.shuffle(outcome_order)
        topic = topic_order[index % len(topic_order)]
        method = method_order[index % len(method_order)]
        outcome = outcome_order[index % len(outcome_order)]
        paragraphs.append(
            f"Memo {index + 1} discusses {topic}. The working group {method}. "
            f"Sample batch {2000 + index} was reviewed by {2 + index % 7} analysts. "
            f"{outcome}\n"
        )
    text = "\n".join(paragraphs)
    if CODE_PATTERN.search(text):
        raise AssertionError("Filler text accidentally contains a code-shaped value")
    return text


def _record_text(record):
    return (
        f"\n[Access record] System {record['key']} uses activation code "
        f"{record['code']}.\n"
    )


def _encode(tokenizer, text):
    return list(tokenizer.encode(text, add_special_tokens=False))


def _allocate_gaps(total, count, rng):
    minimum = MIN_RECORD_GAP_TOKENS * count
    if total < minimum:
        raise ValueError(
            f"Region has {total} filler tokens but needs at least {minimum} "
            "to keep records separated"
        )
    remaining = total - minimum
    gaps = [MIN_RECORD_GAP_TOKENS] * count
    order = list(range(count))
    rng.shuffle(order)
    for index in range(remaining):
        gaps[order[index % count]] += 1
    return gaps


def _pack_region(tokenizer, records, token_budget, rng):
    record_ids = [_encode(tokenizer, _record_text(record)) for record in records]
    filler_budget = token_budget - sum(map(len, record_ids))
    gaps = _allocate_gaps(filler_budget, len(records) + 1, rng)
    filler_text = _make_filler_text(rng)
    filler_ids = _encode(tokenizer, filler_text)
    while len(filler_ids) < filler_budget:
        filler_text += _make_filler_text(rng)
        filler_ids = _encode(tokenizer, filler_text)
    result = []
    cursor = 0
    for index, encoded_record in enumerate(record_ids):
        next_cursor = cursor + gaps[index]
        result.extend(filler_ids[cursor:next_cursor])
        result.extend(encoded_record)
        cursor = next_cursor
    result.extend(filler_ids[cursor:cursor + gaps[-1]])
    if len(result) != token_budget:
        raise AssertionError(f"Packed {len(result)} tokens, expected {token_budget}")
    return result, filler_text


def materialize_scenario(tokenizer, scenario, context_tokens=CONTEXT_TOKENS):
    validate_semantic_scenario(scenario)
    question = f"What is the code for {scenario['target_key']}?"
    messages = [
        {"role": "system", "content": SYSTEM_INSTRUCTION},
        {
            "role": "user",
            "content": f"<DOC>\n{PLACEHOLDER}\n</DOC>\n{question}",
        },
    ]
    template = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    if template.count(PLACEHOLDER) != 1:
        raise ValueError("Chat template did not preserve the document placeholder exactly once")
    prefix_text, suffix_text = template.split(PLACEHOLDER)
    prefix_ids = _encode(tokenizer, prefix_text)
    suffix_ids = _encode(tokenizer, suffix_text)
    if len(suffix_ids) > FINAL_QUERY_WINDOW:
        raise ValueError(
            f"Question/template suffix is {len(suffix_ids)} tokens; maximum is "
            f"{FINAL_QUERY_WINDOW}"
        )
    document_budget = context_tokens - len(prefix_ids) - len(suffix_ids)
    target_record = next(record for record in scenario["records"] if record["is_target"])
    distractors = [record for record in scenario["records"] if not record["is_target"]]
    target_ids = _encode(tokenizer, _record_text(target_record))
    target_start = round(scenario["depth"] * document_budget)
    after_budget = document_budget - target_start - len(target_ids)
    if target_start <= 0 or after_budget <= 0:
        raise ValueError("Target depth leaves no room on one side of the target record")
    rng = random.Random(scenario["seed"] + 17)
    rng.shuffle(distractors)
    before_count = round(scenario["depth"] * len(distractors))
    before_records = distractors[:before_count]
    after_records = distractors[before_count:]
    before_ids, before_filler = _pack_region(
        tokenizer, before_records, target_start, random.Random(scenario["seed"] + 31),
    )
    after_ids, after_filler = _pack_region(
        tokenizer, after_records, after_budget, random.Random(scenario["seed"] + 47),
    )
    document_ids = before_ids + target_ids + after_ids
    input_ids = prefix_ids + document_ids + suffix_ids
    if len(document_ids) != document_budget or len(input_ids) != context_tokens:
        raise AssertionError("Prompt materialization failed the exact-length gate")
    answer = scenario["target_code"]
    leak_text = SYSTEM_INSTRUCTION + suffix_text + before_filler + after_filler
    if answer.casefold() in leak_text.casefold():
        raise AssertionError("Target answer leaked outside its record")
    document_text = tokenizer.decode(document_ids, skip_special_tokens=False)
    extracted_document_codes = [code.upper() for code in CODE_PATTERN.findall(document_text)]
    expected_codes = {record["code"] for record in scenario["records"]}
    if len(extracted_document_codes) != len(scenario["records"]):
        raise AssertionError("A record was truncated or an extra code appeared in the document")
    if set(extracted_document_codes) != expected_codes:
        raise AssertionError("Materialized document codes differ from semantic records")
    if extracted_document_codes.count(answer) != 1:
        raise AssertionError("Target answer must occur exactly once in the document")
    realized_depth = target_start / document_budget
    if abs(realized_depth - scenario["depth"]) > 1 / document_budget:
        raise AssertionError("Target token depth exceeds one-token rounding tolerance")
    return {
        "scenario_id": scenario["scenario_id"],
        "depth": scenario["depth"],
        "target_key": scenario["target_key"],
        "target_code": answer,
        "record_count": len(scenario["records"]),
        "context_tokens": context_tokens,
        "document_tokens": document_budget,
        "target_start_document_token": target_start,
        "target_start_input_token": len(prefix_ids) + target_start,
        "target_record_tokens": len(target_ids),
        "realized_depth": realized_depth,
        "query_window_tokens": len(suffix_ids),
        "input_ids_sha256": token_ids_hash(input_ids),
        "input_ids": input_ids,
    }


def materialize_prompt_manifest(
    tokenizer,
    semantic_manifest,
    tokenizer_id,
    tokenizer_revision,
    context_tokens=CONTEXT_TOKENS,
    include_input_ids=True,
):
    unhashed = {key: value for key, value in semantic_manifest.items() if key != "semantic_sha256"}
    if semantic_manifest.get("semantic_sha256") != canonical_hash(unhashed):
        raise ValueError("Semantic manifest hash is missing or invalid")
    prompts = [
        materialize_scenario(tokenizer, scenario, context_tokens=context_tokens)
        for scenario in semantic_manifest["scenarios"]
    ]
    prompt_set_sha256 = canonical_hash(
        [
            {"scenario_id": prompt["scenario_id"], "sha256": prompt["input_ids_sha256"]}
            for prompt in prompts
        ]
    )
    if not include_input_ids:
        prompts = [
            {key: value for key, value in prompt.items() if key != "input_ids"}
            for prompt in prompts
        ]
    return {
        "manifest_schema": MANIFEST_SCHEMA,
        "generator_version": GENERATOR_VERSION,
        "status": semantic_manifest["status"],
        "semantic_sha256": semantic_manifest["semantic_sha256"],
        "prompt_set_sha256": prompt_set_sha256,
        "tokenizer_id": tokenizer_id,
        "tokenizer_revision": tokenizer_revision,
        "context_tokens": context_tokens,
        "query_window_limit": FINAL_QUERY_WINDOW,
        "prompts": prompts,
    }


def score_response(response, target_code):
    if CODE_PATTERN.fullmatch(target_code) is None:
        raise ValueError(f"Invalid target code: {target_code}")
    extracted = [code.upper() for code in CODE_PATTERN.findall(response)]
    unique_codes = list(dict.fromkeys(extracted))
    target = target_code.upper()
    retrieval_correct = len(unique_codes) == 1 and unique_codes[0] == target
    exact_format = response.strip() == target_code
    legacy_target_anywhere = target.casefold() in response.casefold()
    if retrieval_correct:
        failure_category = "pass"
    elif len(unique_codes) > 1:
        failure_category = "multiple_code"
    elif len(unique_codes) == 1:
        failure_category = "wrong_code"
    elif PARTIAL_CODE_PATTERN.search(response):
        failure_category = "truncated_code"
    else:
        failure_category = "no_code"
    return {
        "retrieval_correct": retrieval_correct,
        "format_correct": exact_format,
        "legacy_target_anywhere": legacy_target_anywhere,
        "failure_category": failure_category,
        "extracted_codes": extracted,
        "unique_extracted_codes": unique_codes,
    }


assert score_response("A123-K45", "A123-K45")["retrieval_correct"]
assert score_response("A123-K", "A123-K45")["failure_category"] == "truncated_code"
assert score_response("none", "A123-K45")["failure_category"] == "no_code"
assert not score_response("A123-K45 B234-L56", "A123-K45")["retrieval_correct"]
print("Frozen generator and strict scorer defined")


Frozen generator and strict scorer defined


## 4. Qwen Prompts, Decode Workload, Schedules, Fingerprint, and Output Root


In [4]:
def atomic_write_text(path, text):
    path = Path(path); tmp = path.with_suffix(path.suffix + f".{os.getpid()}.tmp")
    tmp.write_text(text, encoding="utf-8"); os.replace(tmp, path)

def atomic_write_json(path, value):
    atomic_write_text(path, json.dumps(value, indent=2, ensure_ascii=True) + "\n")

def write_or_verify_json(path, value):
    path = Path(path)
    if path.exists():
        existing = json.loads(path.read_text(encoding="utf-8"))
        assert existing == value, f"Frozen protocol mismatch: {path}"
    else:
        atomic_write_json(path, value)

def tokenizer_identity(t):
    return {
        "backend_sha256": hashlib.sha256(t.backend_tokenizer.to_str().encode()).hexdigest(),
        "chat_template_sha256": hashlib.sha256(t.chat_template.encode()).hexdigest(),
        "special_tokens_sha256": canonical_hash(t.special_tokens_map),
    }

tokenizer_fp16 = AutoTokenizer.from_pretrained(MODEL_FP16, revision=MODEL_FP16_REVISION, trust_remote_code=False)
tokenizer = AutoTokenizer.from_pretrained(MODEL_AWQ, revision=MODEL_AWQ_REVISION, trust_remote_code=False)
identity = tokenizer_identity(tokenizer)
assert tokenizer_identity(tokenizer_fp16) == identity == {
    "backend_sha256": EXPECTED_TOKENIZER_BACKEND_SHA256,
    "chat_template_sha256": EXPECTED_CHAT_TEMPLATE_SHA256,
    "special_tokens_sha256": EXPECTED_SPECIAL_TOKENS_SHA256,
}
assert tokenizer.pad_token_id == 151643 and tokenizer.eos_token_id == 151645
assert tokenizer.bos_token_id is None and tokenizer.unk_token_id is None
assert len(tokenizer) == 151665 and max(tokenizer.get_vocab().values()) == 151664 < VOCAB_SIZE
del tokenizer_fp16
semantic_manifest = build_semantic_manifest()
assert semantic_manifest["semantic_sha256"] == EXPECTED_SEMANTIC_SHA256
prompt_manifest = materialize_prompt_manifest(tokenizer, semantic_manifest, MODEL_AWQ, MODEL_AWQ_REVISION, 4096, True)
assert prompt_manifest["prompt_set_sha256"] == EXPECTED_PROMPT_SET_SHA256
prompt_rows = prompt_manifest["prompts"]
assert len(prompt_rows) == 60 and all(len(row["input_ids"]) == 4096 for row in prompt_rows)
prompts = {row["scenario_id"]: row for row in prompt_rows}
expected_scenario_ids = [f"d{depth}-s{i:02d}" for depth in ("10", "50", "90") for i in range(20)]
sensitivity_scenario_ids = [f"d{depth}-s{i:02d}" for depth in ("10", "50", "90") for i in range(5)]
assert list(prompts) == expected_scenario_ids

base_energy_prompt = prompts["d50-s00"]
suffix_tokens = base_energy_prompt["query_window_tokens"]
ENERGY_PROMPT_IDS = {
    ctx: base_energy_prompt["input_ids"][:ctx - suffix_tokens] + base_energy_prompt["input_ids"][-suffix_tokens:]
    for ctx in CONTEXTS
}
ENERGY_PROMPT_HASHES = {str(ctx): token_ids_hash(ids) for ctx, ids in ENERGY_PROMPT_IDS.items()}
assert ENERGY_PROMPT_HASHES == EXPECTED_ENERGY_PROMPT_HASHES
TRACE_TEXT = (
    "A controlled decode trace keeps the generated workload identical across every compression configuration. "
    "This prevents early end tokens from changing the amount of measured work and supports a fair energy comparison. "
) * 4
TRACE_IDS = tokenizer.encode(TRACE_TEXT, add_special_tokens=False)[:FIXED_DECODE_TOKENS]
TRACE_IDS_SHA256 = token_ids_hash(TRACE_IDS)
TRACE_TEXT_SHA256 = hashlib.sha256(TRACE_TEXT.encode()).hexdigest()
assert len(TRACE_IDS) == 64 and TRACE_IDS_SHA256 == EXPECTED_DECODE_IDS_SHA256
assert TRACE_TEXT_SHA256 == EXPECTED_DECODE_TEXT_SHA256

q0_energy_schedule, awq_energy_schedule = [], []
for block in range(MEASURED_BLOCKS):
    items = [("core", "Q0", ctx) for ctx in CONTEXTS]
    random.Random(408200 + block).shuffle(items)
    q0_energy_schedule.extend({"order": len(q0_energy_schedule), "block": block, "table": t, "variant": v, "context": c} for t, v, c in items)
    items = [("core", v, c) for v in AWQ_CORE_VARIANTS for c in CONTEXTS] + [("sensitivity", v, 4096) for v in SENSITIVITY_VARIANTS]
    random.Random(408300 + block).shuffle(items)
    awq_energy_schedule.extend({"order": len(awq_energy_schedule), "block": block, "table": t, "variant": v, "context": c} for t, v, c in items)
q0_quality_ids = list(expected_scenario_ids); random.Random(408100).shuffle(q0_quality_ids)
q0_quality_schedule = [{"order": i, "table": "core", "scenario_id": sid, "variant": "Q0"} for i, sid in enumerate(q0_quality_ids)]
awq_quality_schedule = []
for scenario_index, sid in enumerate(expected_scenario_ids):
    items = list(AWQ_CORE_VARIANTS); random.Random(408400 + scenario_index).shuffle(items)
    awq_quality_schedule.extend({"order": len(awq_quality_schedule), "table": "core", "scenario_id": sid, "variant": v} for v in items)
    if sid in sensitivity_scenario_ids:
        items = list(SENSITIVITY_VARIANTS); random.Random(408500 + sensitivity_scenario_ids.index(sid)).shuffle(items)
        awq_quality_schedule.extend({"order": len(awq_quality_schedule), "table": "sensitivity", "scenario_id": sid, "variant": v} for v in items)
schedules = {
    "version": "qwen-full-schedule-v1", "q0_energy": q0_energy_schedule,
    "awq_energy": awq_energy_schedule, "q0_quality": q0_quality_schedule,
    "awq_quality": awq_quality_schedule,
}
SCHEDULE_SHA256 = canonical_hash(schedules)
assert (len(q0_energy_schedule), len(awq_energy_schedule), len(q0_quality_schedule), len(awq_quality_schedule)) == (15, 80, 60, 300)
assert SCHEDULE_SHA256 == EXPECTED_SCHEDULE_SHA256

RUNTIME_IDENTITY = {
    "torch": torch.__version__, "cuda_runtime": torch.version.cuda,
    "pandas": pd.__version__, "nvidia_ml_py": importlib.metadata.version("nvidia-ml-py"),
    "python": platform.python_version(), "transformers": transformers.__version__,
    "numpy": np.__version__, "scipy": scipy.__version__,
}
RUNTIME_SHA256 = canonical_hash(RUNTIME_IDENTITY)

ANALYSIS_PROTOCOL = {
    "version": "qwen-full-analysis-v2",
    "energy_metrics": {"total_energy_j": "total_energy_j", "decode_energy_j": "decode_energy_j", "decode_s": "decode_s", "decode_tok_s": "decode_tok_s"},
    "energy_contrasts": [["Q1-W", "Q2"], ["Q1-W", "Q3-Ref"], ["Q1-W", "Q4-Ref"], ["Q2", "Q4-Ref"], ["Q3-Ref", "Q4-Ref"]],
    "energy_family": "five-contrasts-x-four-metrics-holm-20-per-regime",
    "interaction_formula": "(Q4-Ref-Q3-Ref)-(Q2-Q1-W); four metrics; holm-4-per-regime",
    "quality_tests": "paired-mcnemar-exact; bonferroni-12-core-16-sensitivity",
    "pareto": "strict-energy-and-error-dominance-with-one-strict; paired-block-and-depth-scenario-bootstrap-2000",
    "q0_interpretation": "descriptive-cross-phase-reference",
}
ANALYSIS_PROTOCOL_SHA256 = canonical_hash(ANALYSIS_PROTOCOL)

FINGERPRINT_DATA = {
    "schema_version": SCHEMA_VERSION, "model_fp16": MODEL_FP16,
    "model_fp16_revision": MODEL_FP16_REVISION, "model_awq": MODEL_AWQ,
    "model_awq_revision": MODEL_AWQ_REVISION, "transformers": "5.12.1",
    "transformers_commit": TRANSFORMERS_COMMIT,
    "qwen2_source_sha256": SOURCE_HASHES["qwen2"],
    "cache_utils_source_sha256": SOURCE_HASHES["cache_utils"],
    "masking_utils_source_sha256": SOURCE_HASHES["masking_utils"],
    "sdpa_source_sha256": SOURCE_HASHES["sdpa_attention"],
    "numpy": "2.2.6", "scipy": "1.16.1", "gptqmodel": "7.1.0",
    "turboquant": "0.2.0", "turboquant_source_sha256": SOURCE_HASHES["turboquant"],
    "semantic_sha256": EXPECTED_SEMANTIC_SHA256,
    "prompt_set_sha256": EXPECTED_PROMPT_SET_SHA256,
    "tokenizer_backend_sha256": EXPECTED_TOKENIZER_BACKEND_SHA256,
    "chat_template_sha256": EXPECTED_CHAT_TEMPLATE_SHA256,
    "special_tokens_sha256": EXPECTED_SPECIAL_TOKENS_SHA256,
    "energy_prompt_sha256": ENERGY_PROMPT_HASHES,
    "decode_ids_sha256": TRACE_IDS_SHA256, "decode_text_sha256": TRACE_TEXT_SHA256,
    "contexts": list(CONTEXTS), "measured_blocks": MEASURED_BLOCKS,
    "fixed_decode_tokens": FIXED_DECODE_TOKENS, "core_variants": list(CORE_VARIANTS),
    "sensitivity_variants": list(SENSITIVITY_VARIANTS), "expected_counts": EXPECTED_COUNTS,
    "sensitivity_scenarios": sensitivity_scenario_ids,
    "schedule_version": schedules["version"], "schedule_sha256": SCHEDULE_SHA256,
    "pyramid_average_budgets": [512, 1024, 1536], "tq_bits": [2, 3, 4],
    "tq_nmse_max_by_bits": {str(k): v for k, v in TQ_NMSE_MAX_BY_BITS.items()},
    "window": WINDOW, "pool_kernel": POOL_KERNEL, "pyramid_beta": PYRAMID_BETA,
    "parity_atol": PARITY_ATOL, "parity_rtol": PARITY_RTOL,
    "poll_interval_s": POLL_INTERVAL_S, "power_caps_w": list(POWER_CAPS_W),
    "energy_correction": "holm-20",
    "quality_correction": "bonferroni-12-core-16-sensitivity",
    "tq_ref_boundary": "not packed, fused, or physically 4-bit",
    "quality_max_new_tokens": QUALITY_MAX_NEW,
    "architecture": {"layers": NUM_LAYERS, "hidden_size": HIDDEN_SIZE, "intermediate_size": INTERMEDIATE_SIZE,
                     "query_heads": NUM_Q_HEADS, "kv_heads": NUM_KV_HEADS, "head_dim": HEAD_DIM,
                     "vocab_size": VOCAB_SIZE, "max_positions": MAX_POSITIONS, "rope_theta": ROPE_THETA},
    "variant_specs": FINGERPRINT_VARIANT_SPECS,
    "parity_scenarios": ["d10-s00", "d50-s00", "d90-s00"], "probe_scenario": "d50-s00",
    "cache_residual_len": 128,
    "teacher_forced_protocol": "fresh-cache-first-token-rank-plus-continuation-nll-v1",
    "pareto_bootstrap": {"repetitions": 2000, "seed_offset": 99000, "paired_blocks_and_depth_scenarios": True},
    "implementation_gate_version": "qwen-full-adapter-gates-v3",
    "analysis_protocol_sha256": ANALYSIS_PROTOCOL_SHA256,
    "telemetry_gap_gate": {"median_max_s": 0.15, "max_gap_s": 0.5},
}
EXPERIMENT_FINGERPRINT = canonical_hash(FINGERPRINT_DATA)[:16]
assert EXPERIMENT_FINGERPRINT == EXPECTED_FINGERPRINT, (EXPERIMENT_FINGERPRINT, EXPECTED_FINGERPRINT)

from google.colab import drive
drive.mount("/content/drive")
SAVE_DIR = Path("/content/drive/MyDrive") / ARTIFACT_NAME
assert SAVE_DIR.name == ARTIFACT_NAME and SAVE_DIR.name not in FORBIDDEN_ARTIFACT_NAMES
SAVE_DIR.mkdir(parents=True, exist_ok=True)
DIRS = {name: SAVE_DIR / name for name in (
    "core_energy_rows", "sensitivity_energy_rows", "core_quality_rows",
    "sensitivity_quality_rows", "power_traces", "schedules", "analysis",
    "staging", "failed_attempts", "block_commits",
)}
for directory in DIRS.values(): directory.mkdir(exist_ok=True)
ENVIRONMENT_JSON = SAVE_DIR / "qwen_full_environment.json"
VALIDATION_JSON = SAVE_DIR / "qwen_full_validation.json"
FINAL_JSON = DIRS["analysis"] / "final_integrity.json"
if FINAL_JSON.exists():
    final = json.loads(FINAL_JSON.read_text()); assert final.get("passed") and final["fingerprint"] == EXPERIMENT_FINGERPRINT
    raise RuntimeError("This experiment already passed. Treat the artifact directory as immutable.")
if ENVIRONMENT_JSON.exists():
    old = json.loads(ENVIRONMENT_JSON.read_text()); assert old["fingerprint"] == EXPERIMENT_FINGERPRINT and old["schedule_sha256"] == SCHEDULE_SHA256
environment = {**FINGERPRINT_DATA, "fingerprint": EXPERIMENT_FINGERPRINT, "session_id": SESSION_ID, "gpu": GPU.name,
               "vram_gb": round(GPU.total_memory / 2**30, 2), "torch": torch.__version__, "cuda_runtime": torch.version.cuda,
               "power_limit_w": enforced_power_limit_w, "artifact_path": SAVE_DIR.name, "status": "VALIDATION_PENDING"}
write_or_verify_json(SAVE_DIR / "frozen_semantic_manifest.json", semantic_manifest)
write_or_verify_json(SAVE_DIR / "frozen_qwen_prompt_manifest.json", prompt_manifest)
write_or_verify_json(SAVE_DIR / "energy_prompt_manifest.json", {"hashes": ENERGY_PROMPT_HASHES, "ids": ENERGY_PROMPT_IDS, "source_scenario": "d50-s00", "query_suffix_tokens": suffix_tokens})
write_or_verify_json(SAVE_DIR / "fixed_decode_manifest.json", {"text_sha256": TRACE_TEXT_SHA256, "ids_sha256": TRACE_IDS_SHA256, "ids": TRACE_IDS})
write_or_verify_json(SAVE_DIR / "runtime_manifest.json", {"identity": RUNTIME_IDENTITY, "sha256": RUNTIME_SHA256})
write_or_verify_json(SAVE_DIR / "analysis_protocol.json", {"protocol": ANALYSIS_PROTOCOL, "sha256": ANALYSIS_PROTOCOL_SHA256})
write_or_verify_json(DIRS["schedules"] / "measurement_schedules.json", {**schedules, "sha256": SCHEDULE_SHA256})
atomic_write_json(ENVIRONMENT_JSON, environment)
print("Frozen protocol passed:", EXPERIMENT_FINGERPRINT)


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/839 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Mounted at /content/drive
Frozen protocol passed: 861d89617e53034d


## 5. Atomic Shards, Block Promotion, and Strict Resumption


In [5]:
def safe_component(value): return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value))
def row_name(table, variant, context=None, block=None, scenario_id=None):
    fields = [table, variant]
    if context is not None: fields += [f"ctx{context}", f"block{block}"]
    if scenario_id is not None: fields += [scenario_id]
    return "__".join(map(safe_component, fields)) + ".json"

ENERGY_SCHEDULE_MAP = {
    (row["table"], row["variant"], row["context"], row["block"]): row
    for row in q0_energy_schedule + awq_energy_schedule
}
QUALITY_SCHEDULE_MAP = {
    (row["table"], row["variant"], row["scenario_id"]): row
    for row in q0_quality_schedule + awq_quality_schedule
}


def expected_structure(variant, context, decode_tokens):
    spec = VARIANT_SPECS[variant]
    budgets = build_pyramid_budgets(spec["budget"])
    retained = [min(context, budget) for budget in budgets] if spec["eviction"] else [context] * NUM_LAYERS
    final_lengths = [length + decode_tokens for length in retained]
    return {
        "active": NUM_LAYERS * decode_tokens,
        "evicted": sum(context > budget for budget in budgets) if spec["eviction"] else 0,
        "quantized": sum(length > 128 for length in final_lengths) if spec["tq"] else 0,
        "residual_only": [index for index, length in enumerate(final_lengths) if length <= 128] if spec["tq"] else [],
        "cache_min_prefill": min(retained), "cache_max_prefill": max(retained),
    }

def validate_structural_fields(row, prefix, decode_tokens):
    expected = expected_structure(row["variant"], row["context"], decode_tokens)
    key_prefix = f"{prefix}_" if prefix else ""
    assert row[f"{key_prefix}active_decode_cache_updates"] == expected["active"]
    assert row[f"{key_prefix}evicted_layers"] == expected["evicted"]
    assert row[f"{key_prefix}quantized_layers"] == expected["quantized"]
    assert row[f"{key_prefix}residual_only_layers"] == expected["residual_only"]
    nmse = float(row[f"{key_prefix}tq_actual_path_max_nmse"])
    assert math.isfinite(nmse) and nmse >= 0
    if VARIANT_SPECS[row["variant"]]["tq"]: assert nmse <= TQ_NMSE_MAX_BY_BITS[row["tq_bits"]]
    else: assert nmse == 0

def validate_validation_record(record):
    assert record["passed"] is True and record["schema_version"] == SCHEMA_VERSION
    assert record["fingerprint"] == EXPERIMENT_FINGERPRINT and record["runtime_sha256"] == RUNTIME_SHA256
    assert record["source_hashes"] == SOURCE_HASHES
    assert record["fp16_architecture"] == {"passed": True, "layers": NUM_LAYERS, "dtype": "torch.float16"}
    assert record["awq_architecture_passed"] is True
    parity = record["parity_rows"]
    assert [row["scenario_id"] for row in parity] == ["d10-s00", "d50-s00", "d90-s00"]
    assert all(row["passed"] is True and math.isfinite(row["max_abs_logit_difference"])
               and 0 <= row["max_abs_logit_difference"] <= PARITY_ATOL for row in parity)
    probes = record["path_probes"]
    expected_variants = ["Q1-W", "Q2", "Q3-Ref", "Q4-Ref", *SENSITIVITY_VARIANTS]
    assert [probe["variant"] for probe in probes] == expected_variants
    for probe in probes:
        assert probe["structure"]["structural_pass"] is True and probe["storage"]["probe_final_alias_deduplicated"] is True
        expected = expected_structure(probe["variant"], QUALITY_CONTEXT, 1)
        assert probe["structure"]["active_decode_cache_updates"] == expected["active"]
        assert probe["structure"]["evicted_layers"] == expected["evicted"]
        assert probe["structure"]["quantized_layers"] == expected["quantized"]
        assert probe["structure"]["residual_only_layers"] == expected["residual_only"]
        assert probe["structure"]["cache_min_prefill"] == expected["cache_min_prefill"]
        assert probe["structure"]["cache_max_prefill"] == expected["cache_max_prefill"]
        path_nmse = float(probe["structure"]["tq_actual_path_max_nmse"])
        assert math.isfinite(path_nmse) and path_nmse >= 0
        storage = probe["storage"]
        if VARIANT_SPECS[probe["variant"]]["tq"]:
            assert path_nmse <= TQ_NMSE_MAX_BY_BITS[VARIANT_SPECS[probe["variant"]]["bits"]]
            assert storage["probe_final_package_bytes_reconciled"] is True
            assert storage["probe_final_tq_index_bytes"] > 0 and storage["probe_final_tq_norm_bytes"] > 0
            assert storage["probe_final_tq_residual_bytes"] > 0 and storage["probe_final_tq_materialized_bytes"] > 0
        else:
            assert path_nmse == 0
            assert storage["probe_final_tq_index_bytes"] == storage["probe_final_tq_norm_bytes"] == 0
            assert storage["probe_final_tq_residual_bytes"] == storage["probe_final_tq_package_reported_bytes"] == 0
    assert len({probe["top1"] for probe in probes}) == 1
    assert next(probe for probe in probes if probe["variant"] == "Q4-Ref")["storage"]["probe_final_aliased_references"] >= 2
    numerical = record["tq_numerical_probes"]
    assert [probe["bits"] for probe in numerical] == [2, 3, 4]
    assert all(math.isfinite(probe["normalized_mse"]) and 0 <= probe["normalized_mse"] <= TQ_NMSE_MAX_BY_BITS[probe["bits"]]
               and probe["physical_index_bytes_per_coordinate"] == 1 for probe in numerical)
    return record

def validate_common(row):
    assert row["schema_version"] == SCHEMA_VERSION and row["fingerprint"] == EXPERIMENT_FINGERPRINT
    assert row["schedule_sha256"] == SCHEDULE_SHA256 and row["semantic_sha256"] == EXPECTED_SEMANTIC_SHA256
    assert row["prompt_set_sha256"] == EXPECTED_PROMPT_SET_SHA256 and row["structural_pass"] is True
    assert row["runtime_sha256"] == RUNTIME_SHA256
    assert row["variant"] in VARIANT_SPECS and row["variant"] != "Q1-S"
    expected_revision = MODEL_FP16_REVISION if row["variant"] == "Q0" else MODEL_AWQ_REVISION
    assert row["model_revision"] == expected_revision
    spec = VARIANT_SPECS[row["variant"]]
    assert row["average_budget"] == spec["budget"] and row["tq_bits"] == spec["bits"]
    if "block" in row:
        validate_structural_fields(row, "", FIXED_DECODE_TOKENS)
        if spec["tq"]:
            assert row["post_decode_package_bytes_reconciled"] is True
            assert row["post_decode_tq_index_bytes"] > 0 and row["post_decode_tq_norm_bytes"] > 0
            assert row["post_decode_tq_residual_bytes"] > 0 and row["post_decode_tq_materialized_bytes"] > 0
        else:
            assert row["post_decode_tq_index_bytes"] == row["post_decode_tq_norm_bytes"] == 0
            assert row["post_decode_tq_residual_bytes"] == row["post_decode_tq_package_reported_bytes"] == 0
        key = (row["table"], row["variant"], row["context"], row["block"])
        assert key in ENERGY_SCHEDULE_MAP
        item = ENERGY_SCHEDULE_MAP[key]
        assert row["schedule_order"] == item["order"]
        assert row["context"] in CONTEXTS and row["block"] in range(MEASURED_BLOCKS)
        assert row["energy_prompt_sha256"] == ENERGY_PROMPT_HASHES[str(row["context"])]
        assert row["decode_ids_sha256"] == TRACE_IDS_SHA256 and row["fixed_decode_tokens"] == FIXED_DECODE_TOKENS
    else:
        key = (row["table"], row["variant"], row["scenario_id"])
        assert key in QUALITY_SCHEDULE_MAP
        item = QUALITY_SCHEDULE_MAP[key]
        assert row["schedule_order"] == item["order"] and row["context"] == QUALITY_CONTEXT
        assert row["input_ids_sha256"] == prompts[row["scenario_id"]]["input_ids_sha256"]
        assert row["target_code"] == prompts[row["scenario_id"]]["target_code"]

        output_ids = row["output_token_ids"]
        assert isinstance(output_ids, list) and 1 <= len(output_ids) <= QUALITY_MAX_NEW
        assert row["output_tokens"] == len(output_ids)
        assert all(type(token_id) is int and 0 <= token_id < VOCAB_SIZE for token_id in output_ids)
        terminal_ids = {tokenizer.pad_token_id, tokenizer.eos_token_id}
        assert all(token_id not in terminal_ids for token_id in output_ids[:-1])
        if output_ids[-1] in terminal_ids: assert len(output_ids) >= 1
        else: assert len(output_ids) == QUALITY_MAX_NEW
        assert row["response"] == tokenizer.decode(output_ids, skip_special_tokens=True)
        reproduced = score_response(row["response"], row["target_code"])
        for field in ("retrieval_correct", "format_correct", "legacy_target_anywhere", "failure_category"):
            assert row[field] == reproduced[field]
        expected_answer_ids = tokenizer.encode(row["target_code"], add_special_tokens=False)
        assert row["answer_token_ids"] == expected_answer_ids and len(expected_answer_ids) >= 2
        assert row["teacher_forced_structural_pass"] is True
        generation_decode_tokens = max(0, row["output_tokens"] - 1)
        teacher_decode_tokens = len(row["answer_token_ids"]) - 1
        validate_structural_fields(row, "generation", generation_decode_tokens)
        validate_structural_fields(row, "teacher_forced", teacher_decode_tokens)
        if row["variant"] == "Q0":
            assert row["generation_stock_cache_validated"] is True and row["teacher_forced_stock_cache_validated"] is True
        else:
            assert row["generation_stock_cache_validated"] is False and row["teacher_forced_stock_cache_validated"] is False

def read_shards(directory):
    rows = []
    for path in sorted(directory.glob("*.json")):
        row = json.loads(path.read_text()); validate_common(row)
        if "block" in row:
            expected_name = row_name(row["table"], row["variant"], row["context"], row["block"])
        else:
            expected_name = row_name(row["table"], row["variant"], scenario_id=row["scenario_id"])
        assert path.name == expected_name
        assert directory == DIRS[f"{row['table']}_{'energy' if 'block' in row else 'quality'}_rows"]
        rows.append(row)
    return rows

def refresh_csv(directory, csv_name):
    rows = read_shards(directory)
    if rows:
        tmp = (SAVE_DIR / csv_name).with_suffix(".csv.tmp")
        pd.DataFrame(rows).to_csv(tmp, index=False); os.replace(tmp, SAVE_DIR / csv_name)
    return rows

def commit_quality_row(row, table):
    validate_common(row)
    directory = DIRS[f"{table}_quality_rows"]
    path = directory / row_name(table, row["variant"], scenario_id=row["scenario_id"])
    if path.exists():
        old = json.loads(path.read_text()); validate_common(old); assert old == row
    else: atomic_write_json(path, row)


def integrate_trace_phase(samples, cap=None):
    times = np.asarray([sample["elapsed_s"] for sample in samples], dtype=float)
    powers = np.asarray([sample["power_w"] for sample in samples], dtype=float)
    assert len(times) >= 2 and np.all(np.diff(times) >= 0)
    assert np.isfinite(powers).all() and (powers >= 0).all()
    if len(times) > 2:
        gaps = np.diff(times); assert np.median(gaps) <= 0.15 and gaps.max() <= 0.5
    return float(np.trapezoid(np.minimum(powers, cap) if cap is not None else powers, times))

def validate_trace_and_row(row, trace_path):
    validate_common(row)
    assert trace_path.name == row["trace_name"] and trace_path.exists()
    assert hashlib.sha256(trace_path.read_bytes()).hexdigest() == row["trace_sha256"]
    trace = json.loads(trace_path.read_text(encoding="utf-8"))
    for key in ("schema_version", "fingerprint", "table", "variant", "context", "block", "decode_ids_sha256"):
        expected = row["decode_ids_sha256"] if key == "decode_ids_sha256" else row[key]
        assert trace[key] == expected, (key, trace[key], expected)
    values = {}
    for regime, cap in (("raw", None), ("cap82", 82.0), ("cap70", 70.0)):
        values[f"{regime}_prefill"] = integrate_trace_phase(trace["prefill"], cap)
        values[f"{regime}_decode"] = integrate_trace_phase(trace["decode"], cap)
        values[f"{regime}_total"] = values[f"{regime}_prefill"] + values[f"{regime}_decode"]
    comparisons = {
        "prefill_energy_j": values["raw_prefill"], "decode_energy_j": values["raw_decode"],
        "total_energy_j": values["raw_total"], "cap82_prefill_energy_j": values["cap82_prefill"],
        "cap82_decode_energy_j": values["cap82_decode"], "cap82_total_energy_j": values["cap82_total"],
        "cap70_prefill_energy_j": values["cap70_prefill"], "cap70_decode_energy_j": values["cap70_decode"],
        "cap70_total_energy_j": values["cap70_total"],
    }
    assert all(np.isclose(row[key], value, atol=1e-6, rtol=1e-9) for key, value in comparisons.items())
    return values

def block_marker(phase, block): return DIRS["block_commits"] / f"{phase}_block{block}.json"

def expected_block_items(phase, block):
    schedule = q0_energy_schedule if phase == "q0" else awq_energy_schedule
    return [row for row in schedule if row["block"] == block]

def canonical_energy_paths(item):
    row_dir = DIRS[f"{item['table']}_energy_rows"]
    name = row_name(item["table"], item["variant"], item["context"], item["block"])
    trace_name = name.replace(".json", "__trace.json")
    return row_dir / name, DIRS["power_traces"] / trace_name

def recover_incomplete_block(phase, block):
    marker = block_marker(phase, block)
    items = expected_block_items(phase, block)
    canonical = [p for item in items for p in canonical_energy_paths(item)]
    present = [p for p in canonical if p.exists()]
    if marker.exists():
        record = json.loads(marker.read_text())
        assert record["fingerprint"] == EXPERIMENT_FINGERPRINT and record["phase"] == phase and record["block"] == block
        assert len(record["items"]) == len(items) and len(present) == 2 * len(items)
        expected_names = []
        for item in items:
            row_path, trace_path = canonical_energy_paths(item)
            row = json.loads(row_path.read_text()); validate_trace_and_row(row, trace_path)
            expected_names.append({"row": row_path.name, "trace": trace_path.name})
        assert record["items"] == expected_names
        return True
    if present:
        failed = DIRS["failed_attempts"] / f"recovered_{phase}_block{block}_{time.strftime('%Y%m%dT%H%M%S')}"
        failed.mkdir()
        for p in present: shutil.move(str(p), failed / p.name)
    return False

def promote_energy_block(phase, block, stage, items):
    assert len(list(stage.glob("*__row.json"))) == len(items)
    assert len(list(stage.glob("*__trace.json"))) == len(items)
    prepared = []
    for item in items:
        stem = row_name(item["table"], item["variant"], item["context"], item["block"]).removesuffix(".json")
        staged_row, staged_trace = stage / f"{stem}__row.json", stage / f"{stem}__trace.json"
        row = json.loads(staged_row.read_text()); validate_common(row)
        assert row["trace_name"] == staged_trace.name
        validate_trace_and_row(row, staged_trace)
        row_path, trace_path = canonical_energy_paths(item)
        assert not row_path.exists() and not trace_path.exists()
        prepared.append((row, staged_row, staged_trace, row_path, trace_path))
    promoted = []
    for row, staged_row, staged_trace, row_path, trace_path in prepared:
        os.replace(staged_trace, trace_path)
        assert hashlib.sha256(trace_path.read_bytes()).hexdigest() == row["trace_sha256"]
        atomic_write_json(row_path, row)
        validate_trace_and_row(json.loads(row_path.read_text()), trace_path)
        promoted.append({"row": row_path.name, "trace": trace_path.name})
    atomic_write_json(block_marker(phase, block), {"fingerprint": EXPERIMENT_FINGERPRINT, "phase": phase, "block": block,
                      "attempt_id": stage.name, "items": promoted})
    shutil.rmtree(stage)
    refresh_csv(DIRS["core_energy_rows"], "core_energy_results.csv")
    refresh_csv(DIRS["sensitivity_energy_rows"], "sensitivity_energy_results.csv")


## 6. Generalized Validated Qwen Adapter and Cache Audit


In [6]:
def validate_config(config, quantized):
    assert config.model_type == "qwen2" and config.architectures == ["Qwen2ForCausalLM"]
    assert (config.num_hidden_layers, config.hidden_size, config.intermediate_size) == (36, 2048, 11008)
    assert (config.num_attention_heads, config.num_key_value_heads) == (16, 2)
    assert getattr(config, "head_dim", 128) == 128 and config.vocab_size == VOCAB_SIZE
    assert config.max_position_embeddings == MAX_POSITIONS and config.sliding_window is None
    assert config.attention_dropout == 0.0 and config.tie_word_embeddings is True
    assert list(config.layer_types) == ["full_attention"] * NUM_LAYERS
    assert float(config.rope_parameters["rope_theta"]) == ROPE_THETA
    if quantized:
        q = config.quantization_config.to_dict() if hasattr(config.quantization_config, "to_dict") else dict(config.quantization_config)
        assert q["quant_method"] == "awq" and q["bits"] == 4 and q["group_size"] == 128 and q["zero_point"] is True
        assert str(q.get("format", q.get("version"))).lower().endswith("gemm")

def clear_gpu(): gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
def release_model():
    old = globals().pop("model", None)
    if old is not None: del old
    clear_gpu()

ORIGINAL_QWEN_FORWARD = Qwen2Attention.forward
def validate_model(model, quantized):
    validate_config(model.config, quantized)
    assert len(model.model.layers) == NUM_LAYERS
    assert model.model.embed_tokens.num_embeddings == VOCAB_SIZE
    for i, layer in enumerate(model.model.layers):
        a = layer.self_attn; assert isinstance(a, Qwen2Attention) and a.layer_idx == i
        assert a.num_key_value_groups == 8 and a.head_dim == 128 and a.sliding_window is None
        assert a.layer_type == "full_attention"
        assert a.q_proj.bias is not None and a.k_proj.bias is not None and a.v_proj.bias is not None and a.o_proj.bias is None
        if quantized:
            assert all(not isinstance(p, torch.nn.Linear) for p in (a.q_proj, a.k_proj, a.v_proj, a.o_proj))
    assert next(model.parameters()).device.type == "cuda"
    if not quantized: assert model.model.embed_tokens.weight.dtype == torch.float16

def load_fp16():
    assert Qwen2Attention.forward is ORIGINAL_QWEN_FORWARD
    m = AutoModelForCausalLM.from_pretrained(MODEL_FP16, revision=MODEL_FP16_REVISION, dtype=torch.float16,
         device_map={"": 0}, attn_implementation="sdpa", trust_remote_code=False).eval()
    validate_model(m, False); return m

def load_awq():
    assert Qwen2Attention.forward is ORIGINAL_QWEN_FORWARD
    m = AutoModelForCausalLM.from_pretrained(MODEL_AWQ, revision=MODEL_AWQ_REVISION,
         device_map={"": 0}, attn_implementation="sdpa", trust_remote_code=False).eval()
    validate_model(m, True); return m

VARIANT_SPECS = {
    "Q0": {"wrapper": False, "eviction": False, "budget": 1024, "tq": False, "bits": 4},
    "Q1-S": {"wrapper": False, "eviction": False, "budget": 1024, "tq": False, "bits": 4},
    "Q1-W": {"wrapper": True, "eviction": False, "budget": 1024, "tq": False, "bits": 4},
    "Q2": {"wrapper": True, "eviction": True, "budget": 1024, "tq": False, "bits": 4},
    "Q3-Ref": {"wrapper": True, "eviction": False, "budget": 1024, "tq": True, "bits": 4},
    "Q4-Ref": {"wrapper": True, "eviction": True, "budget": 1024, "tq": True, "bits": 4},
    "Q2-B512": {"wrapper": True, "eviction": True, "budget": 512, "tq": False, "bits": 4},
    "Q2-B1536": {"wrapper": True, "eviction": True, "budget": 1536, "tq": False, "bits": 4},
    "Q3-Ref-TQ2": {"wrapper": True, "eviction": False, "budget": 1024, "tq": True, "bits": 2},
    "Q3-Ref-TQ3": {"wrapper": True, "eviction": False, "budget": 1024, "tq": True, "bits": 3},
}
assert VARIANT_SPECS == FINGERPRINT_VARIANT_SPECS
ACTIVE_SPEC, ACTIVE_CONTEXT, ACTIVE_BUDGETS = None, None, None
PKV_LOG, UPDATE_LOG, POSITION_LOG, ATTENTION_LOG, TQ_PATH_LOG, EVENT_LOG = [], [], [], [], [], []

def build_pyramid_budgets(average_budget):
    old_average = int(average_budget) - WINDOW; last = round(old_average / PYRAMID_BETA); first = 2 * old_average - last
    budgets = (np.linspace(first, last, NUM_LAYERS).round().astype(int) + WINDOW).tolist()
    assert len(budgets) == NUM_LAYERS and sum(budgets) == NUM_LAYERS * int(average_budget)
    assert min(budgets) >= WINDOW and all(a >= b for a, b in zip(budgets, budgets[1:])); return budgets

def clear_adapter_logs():
    for log in (PKV_LOG, UPDATE_LOG, POSITION_LOG, ATTENTION_LOG, TQ_PATH_LOG, EVENT_LOG): log.clear()

def select_pyramid_kv(q, k, v, layer_idx, groups, scaling):
    cache_len = k.shape[-2]; budget = min(ACTIVE_BUDGETS[layer_idx], cache_len)
    if cache_len <= budget:
        PKV_LOG.append({"layer": layer_idx, "before": cache_len, "after": cache_len, "budget": budget, "evicted": False})
        EVENT_LOG.append({"phase": "prefill", "layer": layer_idx, "event": "selection"})
        return k, v
    obs = min(WINDOW, cache_len); old_len = cache_len - obs; old_keep = budget - obs
    recent_q = q[:, :, -obs:, :]; repeated_k = repeat_kv(k, groups)
    scores = torch.matmul(recent_q, repeated_k.transpose(2, 3)) * scaling
    causal = torch.triu(torch.full((obs, obs), float("-inf"), device=scores.device, dtype=scores.dtype), diagonal=1)
    scores[:, :, :, -obs:] += causal[None, None]
    probs = F.softmax(scores, dim=-1, dtype=torch.float32)
    prefix = probs[:, :, :, :old_len].sum(-2).reshape(probs.shape[0], NUM_KV_HEADS, groups, old_len).mean(2)
    prefix = F.avg_pool1d(prefix, POOL_KERNEL, 1, POOL_KERNEL // 2)
    topk = prefix.topk(old_keep, dim=-1).indices.sort(dim=-1).values
    gather = topk.unsqueeze(-1).expand(-1, -1, -1, HEAD_DIM)
    selected_k = torch.cat([k[:, :, :old_len].gather(2, gather), k[:, :, -obs:]], dim=2).contiguous()
    selected_v = torch.cat([v[:, :, :old_len].gather(2, gather), v[:, :, -obs:]], dim=2).contiguous()
    PKV_LOG.append({"layer": layer_idx, "before": cache_len, "after": selected_k.shape[-2], "budget": budget,
                    "evicted": True, "head_selection_differs": bool((topk[:, 1:] != topk[:, :-1]).any().item())})
    EVENT_LOG.append({"phase": "prefill", "layer": layer_idx, "event": "selection"})
    return selected_k, selected_v

def qwen_full_wrapper(self, hidden_states, position_embeddings, attention_mask, past_key_values=None, **kwargs):
    shape = hidden_states.shape[:-1]; hidden_shape = (*shape, -1, self.head_dim)
    q = self.q_proj(hidden_states).view(hidden_shape).transpose(1, 2)
    k = self.k_proj(hidden_states).view(hidden_shape).transpose(1, 2)
    v = self.v_proj(hidden_states).view(hidden_shape).transpose(1, 2)
    cos, sin = position_embeddings; q, k = apply_rotary_pos_emb(q, k, cos, sin)
    positions = kwargs.get("position_ids"); assert positions is not None
    input_tokens = k.shape[-2]; assert positions.shape == (1, input_tokens)
    is_prefill = past_key_values is not None and past_key_values.get_seq_length(self.layer_idx) == 0 and input_tokens == ACTIVE_CONTEXT
    phase = "prefill" if is_prefill else "decode"
    if is_prefill: assert int(positions[0, 0]) == 0 and int(positions[0, -1]) == ACTIVE_CONTEXT - 1
    else: assert input_tokens == 1 and int(positions[0, 0]) >= ACTIVE_CONTEXT and attention_mask is None
    POSITION_LOG.append({"phase": phase, "layer": self.layer_idx, "start": int(positions[0, 0]), "end": int(positions[0, -1])})
    expected_ptr = None
    if past_key_values is not None and not is_prefill:
        k, v = past_key_values.update(k, v, self.layer_idx); expected_ptr = (int(k.data_ptr()), int(v.data_ptr()))
        UPDATE_LOG.append({"phase": "decode", "layer": self.layer_idx, "returned_tokens": k.shape[-2], "position": int(positions[0, 0])})
    interface = ALL_ATTENTION_FUNCTIONS.get_interface(self.config._attn_implementation, eager_attention_forward)
    observed_ptr = (int(k.data_ptr()), int(v.data_ptr()))
    if expected_ptr is not None: assert observed_ptr == expected_ptr
    mask_id = id(attention_mask)
    ATTENTION_LOG.append({"phase": phase, "layer": self.layer_idx, "key_tokens": k.shape[-2], "returned_cache_used": expected_ptr is None or observed_ptr == expected_ptr,
                          "mask_id": mask_id, "position_start": int(positions[0, 0]), "position_end": int(positions[0, -1])})
    attn_output, attn_weights = interface(self, q, k, v, attention_mask,
        dropout=0.0 if not self.training else self.attention_dropout, scaling=self.scaling,
        sliding_window=self.sliding_window, **kwargs)
    assert id(attention_mask) == mask_id
    EVENT_LOG.append({"phase": phase, "layer": self.layer_idx, "event": "attention"})
    if is_prefill:
        stored_k, stored_v = (select_pyramid_kv(q, k, v, self.layer_idx, self.num_key_value_groups, self.scaling)
                              if ACTIVE_SPEC["eviction"] else (k, v))
        original_k, original_v = stored_k, stored_v
        returned_k, returned_v = past_key_values.update(stored_k, stored_v, self.layer_idx)
        EVENT_LOG.append({"phase": "prefill", "layer": self.layer_idx, "event": "cache_update"})
        UPDATE_LOG.append({"phase": "prefill", "layer": self.layer_idx, "attention_tokens": k.shape[-2],
                           "update_input_tokens": stored_k.shape[-2], "returned_tokens": returned_k.shape[-2]})
        if ACTIVE_SPEC["tq"]:
            quantized = max(stored_k.shape[-2] - 128, 0)
            def nmse(a, b):
                if not quantized: return 0.0
                return float(((a[..., :quantized].float() - b[..., :quantized].float()).square().sum(-1) /
                              b[..., :quantized].float().square().sum(-1).clamp_min(1e-12)).mean())
            key_nmse, value_nmse = nmse(returned_k, original_k), nmse(returned_v, original_v)
            residual = min(stored_k.shape[-2], 128)
            assert torch.equal(returned_k[..., -residual:, :], original_k[..., -residual:, :])
            assert torch.equal(returned_v[..., -residual:, :], original_v[..., -residual:, :])
            assert max(key_nmse, value_nmse) <= TQ_NMSE_MAX_BY_BITS[ACTIVE_SPEC["bits"]]
            TQ_PATH_LOG.append({"layer": self.layer_idx, "quantized_tokens": quantized,
                                "key_nmse": key_nmse, "value_nmse": value_nmse, "residual_exact": True})
    return self.o_proj(attn_output.reshape(*shape, -1).contiguous()), attn_weights

def enable_variant(variant, context):
    global ACTIVE_SPEC, ACTIVE_CONTEXT, ACTIVE_BUDGETS
    assert Qwen2Attention.forward is ORIGINAL_QWEN_FORWARD
    spec = VARIANT_SPECS[variant]; assert spec["wrapper"]
    ACTIVE_SPEC, ACTIVE_CONTEXT = dict(spec), int(context)
    ACTIVE_BUDGETS = build_pyramid_budgets(spec["budget"])
    clear_adapter_logs(); Qwen2Attention.forward = qwen_full_wrapper

def disable_variant():
    global ACTIVE_SPEC, ACTIVE_CONTEXT, ACTIVE_BUDGETS
    Qwen2Attention.forward = ORIGINAL_QWEN_FORWARD
    ACTIVE_SPEC = ACTIVE_CONTEXT = ACTIVE_BUDGETS = None

def new_cache(model, variant):
    spec = VARIANT_SPECS[variant]
    return TurboQuantCache(bits=spec["bits"]) if spec["tq"] else DynamicCache(config=model.config)

def cache_lengths(cache):
    return [int(layer._total_len) if hasattr(layer, "_total_len") else int(layer.get_seq_length()) for layer in cache.layers]

def named_cache_tensors(cache):
    def walk(name, value):
        if isinstance(value, torch.Tensor):
            yield name, value
        elif isinstance(value, Mapping):
            for child_name, child in value.items(): yield from walk(f"{name}.{child_name}", child)
        elif isinstance(value, (list, tuple)):
            for index, child in enumerate(value): yield from walk(f"{name}.{index}", child)
    for i, layer in enumerate(cache.layers):
        for name, value in vars(layer).items():
            for nested_name, tensor in walk(name, value): yield i, nested_name, tensor

def storage_summary(tensors):
    refs, unique = 0, {}
    for t in tensors:
        if t.numel() == 0: continue
        refs += 1; s = t.untyped_storage(); unique[(str(t.device), int(s.data_ptr()), int(s.nbytes()))] = int(s.nbytes())
    return sum(unique.values()), len(unique), refs

def cache_storage_metrics(cache, prefix):
    named = list(named_cache_tensors(cache)); all_t = [t for _, _, t in named]
    pick = lambda names: [t for _, n, t in named if n in names]
    indices = pick({"_key_indices", "_value_indices"}); norms = pick({"_key_norms", "_value_norms"})
    residuals = pick({"_residual_keys", "_residual_values"}); materialized = pick({"keys", "values"})
    live, unique, refs = storage_summary(all_t)
    raw_reference_bytes = sum(int(t.untyped_storage().nbytes()) for t in all_t if t.numel() > 0)
    payload_logical = sum(t.numel() * t.element_size() for t in indices + norms + residuals)
    package = int(cache.memory_usage_bytes().get("total_bytes", 0)) if callable(getattr(cache, "memory_usage_bytes", None)) else 0
    return {f"{prefix}_cache_live_bytes": live, f"{prefix}_unique_storages": unique,
            f"{prefix}_tensor_references": refs, f"{prefix}_aliased_references": refs - unique,
            f"{prefix}_raw_reference_bytes": raw_reference_bytes,
            f"{prefix}_alias_deduplicated": live <= raw_reference_bytes,
            f"{prefix}_retained_fp16_bytes": int(sum(cache_lengths(cache)) * NUM_KV_HEADS * HEAD_DIM * 4),
            f"{prefix}_tq_index_bytes": storage_summary(indices)[0], f"{prefix}_tq_norm_bytes": storage_summary(norms)[0],
            f"{prefix}_tq_residual_bytes": storage_summary(residuals)[0], f"{prefix}_tq_materialized_bytes": storage_summary(materialized)[0],
            f"{prefix}_tq_package_reported_bytes": package, f"{prefix}_logical_tq_payload_bytes": payload_logical,
            f"{prefix}_package_bytes_reconciled": package == payload_logical}


TQ_STATE_NAMES = {"_key_indices", "_value_indices", "_key_norms", "_value_norms", "_residual_keys", "_residual_values"}

def validate_stock_cache(cache, context, prefill_lengths, decode_tokens):
    assert isinstance(cache, DynamicCache) and not isinstance(cache, TurboQuantCache)
    assert len(cache.layers) == NUM_LAYERS
    assert prefill_lengths == [context] * NUM_LAYERS
    assert cache_lengths(cache) == [context + decode_tokens] * NUM_LAYERS
    assert all(name not in TQ_STATE_NAMES for _, name, _ in named_cache_tensors(cache))
    return {"structural_pass": True, "quantized_layers": 0, "residual_only_layers": [],
            "evicted_layers": 0, "cache_min_prefill": context, "cache_max_prefill": context,
            "active_decode_cache_updates": NUM_LAYERS * decode_tokens,
            "tq_actual_path_max_nmse": 0.0, "stock_cache_validated": True}

def validate_cache_path(cache, variant, context, prefill_lengths, decode_tokens):
    spec = VARIANT_SPECS[variant]; budgets = build_pyramid_budgets(spec["budget"])
    expected = [min(context, b) for b in budgets] if spec["eviction"] else [context] * NUM_LAYERS
    assert prefill_lengths == expected and cache_lengths(cache) == [n + decode_tokens for n in expected]
    assert len(UPDATE_LOG) == NUM_LAYERS * (1 + decode_tokens)
    assert len(POSITION_LOG) == NUM_LAYERS * (1 + decode_tokens)
    prefill_positions = [r for r in POSITION_LOG if r["phase"] == "prefill"]
    assert len(prefill_positions) == NUM_LAYERS and all(r["start"] == 0 and r["end"] == context - 1 for r in prefill_positions)
    for layer in range(NUM_LAYERS):
        decode_positions = [r["start"] for r in POSITION_LOG if r["phase"] == "decode" and r["layer"] == layer]
        assert decode_positions == list(range(context, context + decode_tokens))
        events = [r["event"] for r in EVENT_LOG if r["phase"] == "prefill" and r["layer"] == layer]
        expected_events = ["attention", "selection", "cache_update"] if spec["eviction"] else ["attention", "cache_update"]
        assert events == expected_events
    assert all(row["attention_tokens"] == context for row in UPDATE_LOG if row["phase"] == "prefill")
    assert [row["update_input_tokens"] for row in UPDATE_LOG if row["phase"] == "prefill"] == expected
    assert len(ATTENTION_LOG) == NUM_LAYERS * (1 + decode_tokens)
    expected_attention = []
    for layer in range(NUM_LAYERS):
        expected_attention.append(("prefill", layer, context, 0, context - 1))
    for step in range(decode_tokens):
        for layer, retained in enumerate(expected):
            expected_attention.append(("decode", layer, retained + step + 1, context + step, context + step))
    observed_attention = [(row["phase"], row["layer"], row["key_tokens"], row["position_start"], row["position_end"])
                          for row in ATTENTION_LOG]
    assert observed_attention == expected_attention
    assert all(type(row["mask_id"]) is int and row["mask_id"] > 0 for row in ATTENTION_LOG)
    assert all(row["returned_cache_used"] for row in ATTENTION_LOG)
    if spec["eviction"]:
        assert len(PKV_LOG) == NUM_LAYERS and [row["after"] for row in PKV_LOG] == expected
    else: assert not PKV_LOG
    quantized_layers, residual_only = 0, []
    if spec["tq"]:
        assert isinstance(cache, TurboQuantCache) and cache.bits == spec["bits"]
        assert len(TQ_PATH_LOG) == NUM_LAYERS and all(row["residual_exact"] for row in TQ_PATH_LOG)
        for i, (layer, total) in enumerate(zip(cache.layers, cache_lengths(cache))):
            assert layer.bits == spec["bits"] and layer.residual_len == 128
            quantized = max(total - layer.residual_len, 0); residual = min(total, layer.residual_len)
            if quantized:
                quantized_layers += 1
                for indices in (layer._key_indices, layer._value_indices):
                    assert indices.dtype == torch.uint8 and int(indices.max()) < 2 ** spec["bits"]
                    assert indices.shape == (1, NUM_KV_HEADS, quantized, HEAD_DIM)
                for norms in (layer._key_norms, layer._value_norms):
                    assert norms.shape == (1, NUM_KV_HEADS, quantized, 1)
                    assert torch.isfinite(norms).all() and (norms >= 0).all()
            else:
                residual_only.append(i)
                assert layer._key_indices.numel() == layer._value_indices.numel() == 0
                assert layer._key_norms.numel() == layer._value_norms.numel() == 0
            assert layer._residual_keys.shape[-2] == layer._residual_values.shape[-2] == residual
            assert layer.keys.shape[-2] == layer.values.shape[-2] == total
            assert torch.isfinite(layer.keys).all() and torch.isfinite(layer.values).all()
        assert quantized_layers == sum(n > 128 for n in cache_lengths(cache))
        metrics = cache_storage_metrics(cache, "validation")
        assert metrics["validation_package_bytes_reconciled"]
        assert metrics["validation_tq_index_bytes"] > 0 and metrics["validation_tq_norm_bytes"] > 0
        assert metrics["validation_tq_residual_bytes"] > 0 and metrics["validation_tq_materialized_bytes"] > 0
    else:
        assert not isinstance(cache, TurboQuantCache) and not TQ_PATH_LOG
        assert all(name not in TQ_STATE_NAMES for _, name, _ in named_cache_tensors(cache))
    return {"structural_pass": True, "quantized_layers": quantized_layers, "residual_only_layers": residual_only,
            "evicted_layers": sum(context > b for b in budgets) if spec["eviction"] else 0,
            "cache_min_prefill": min(expected), "cache_max_prefill": max(expected),
            "active_decode_cache_updates": NUM_LAYERS * decode_tokens,
            "tq_actual_path_max_nmse": max([max(x["key_nmse"], x["value_nmse"]) for x in TQ_PATH_LOG] or [0.0])}


## 7. Mandatory Architecture, Parity, and All-Path Validation

No measurement cell may run unless this gate passes.


In [7]:
def prompt_tensor(prompt, model):
    ids = torch.tensor([prompt["input_ids"]], dtype=torch.long, device=model.device)
    assert token_ids_hash(ids[0].tolist()) == prompt["input_ids_sha256"]; return ids

def run_generation(model, variant, prompt, max_new=QUALITY_MAX_NEW):
    spec = VARIANT_SPECS[variant]; clear_adapter_logs()
    if spec["wrapper"]: enable_variant(variant, 4096)
    else: assert Qwen2Attention.forward is ORIGINAL_QWEN_FORWARD
    cache = new_cache(model, variant); ids = prompt_tensor(prompt, model); generated = []
    try:
        with torch.inference_mode():
            out = model(input_ids=ids, position_ids=torch.arange(4096, device=model.device).unsqueeze(0),
                        past_key_values=cache, use_cache=True, logits_to_keep=1)
            prefill_lengths = cache_lengths(cache); logits = out.logits[0, -1].float().cpu()
            token = out.logits[:, -1].argmax(-1, keepdim=True)
            for step in range(max_new):
                generated.append(token)
                if int(token) in {151643, 151645} or step == max_new - 1: break
                position = torch.tensor([[4096 + step]], device=model.device)
                out = model(input_ids=token, position_ids=position, attention_mask={"full_attention": None},
                            past_key_values=cache, use_cache=True, logits_to_keep=1)
                token = out.logits[:, -1].argmax(-1, keepdim=True)
        output_ids = torch.cat(generated, 1)[0].tolist()
        decode_tokens = max(0, len(output_ids) - 1)
        structure = (validate_cache_path(cache, variant, 4096, prefill_lengths, decode_tokens)
                     if spec["wrapper"] else validate_stock_cache(cache, 4096, prefill_lengths, decode_tokens))
        return {"logits": logits, "top1": int(logits.argmax()), "output_ids": output_ids,
                "response": tokenizer.decode(output_ids, skip_special_tokens=True), "prefill_lengths": prefill_lengths,
                "final_lengths": cache_lengths(cache), "structure": structure}
    finally:
        if spec["wrapper"]: disable_variant()
        del cache, ids; clear_adapter_logs(); clear_gpu()

def run_one_token_probe(model, variant, prompt):
    enable_variant(variant, 4096); cache = new_cache(model, variant); ids = prompt_tensor(prompt, model)
    answer_ids = tokenizer.encode(prompt["target_code"], add_special_tokens=False)
    try:
        with torch.inference_mode():
            out = model(input_ids=ids, position_ids=torch.arange(4096, device=model.device).unsqueeze(0),
                        past_key_values=cache, use_cache=True, logits_to_keep=1)
            lengths = cache_lengths(cache); logits = out.logits[0, -1].float().cpu()
            token = torch.tensor([[answer_ids[0]]], device=model.device)
            model(input_ids=token, position_ids=torch.tensor([[4096]], device=model.device),
                  attention_mask={"full_attention": None}, past_key_values=cache, use_cache=True, logits_to_keep=1)
        structure = validate_cache_path(cache, variant, 4096, lengths, 1)
        return {"variant": variant, "logits": logits, "top1": int(logits.argmax()), "structure": structure,
                "storage": cache_storage_metrics(cache, "probe_final")}
    finally:
        disable_variant(); del cache, ids; clear_adapter_logs(); clear_gpu()

VALIDATION_JSON.unlink(missing_ok=True)
model = load_fp16(); fp16_gate = {"passed": True, "layers": len(model.model.layers), "dtype": str(model.model.embed_tokens.weight.dtype)}; release_model()
model = load_awq(); parity_rows, probes = [], []
try:
    for sid in ("d10-s00", "d50-s00", "d90-s00"):
        stock = run_generation(model, "Q1-S", prompts[sid]); wrapped = run_generation(model, "Q1-W", prompts[sid])
        max_abs = float((stock["logits"] - wrapped["logits"]).abs().max())
        assert torch.allclose(stock["logits"], wrapped["logits"], atol=PARITY_ATOL, rtol=PARITY_RTOL)
        assert stock["top1"] == wrapped["top1"] and stock["output_ids"] == wrapped["output_ids"] and stock["response"] == wrapped["response"]
        assert stock["prefill_lengths"] == wrapped["prefill_lengths"] == [4096] * NUM_LAYERS
        assert stock["final_lengths"] == wrapped["final_lengths"]
        parity_rows.append({"scenario_id": sid, "max_abs_logit_difference": max_abs, "passed": True})
    for variant in ("Q1-W", "Q2", "Q3-Ref", "Q4-Ref", *SENSITIVITY_VARIANTS):
        probes.append(run_one_token_probe(model, variant, prompts["d50-s00"]))
    baseline = probes[0]["logits"]
    assert all(torch.allclose(baseline, p["logits"], atol=PARITY_ATOL, rtol=PARITY_RTOL) for p in probes[1:])
    assert all(p["top1"] == probes[0]["top1"] for p in probes[1:])
    serializable = [{k: v for k, v in p.items() if k != "logits"} for p in probes]
    assert all(p["storage"]["probe_final_alias_deduplicated"] for p in serializable)
    q4_probe = next(p for p in serializable if p["variant"] == "Q4-Ref")
    assert q4_probe["storage"]["probe_final_aliased_references"] >= 2
    for variant in ("Q3-Ref", "Q4-Ref", "Q3-Ref-TQ2", "Q3-Ref-TQ3"):
        probe = next(p for p in serializable if p["variant"] == variant)
        assert probe["storage"]["probe_final_package_bytes_reconciled"]
        assert probe["storage"]["probe_final_tq_index_bytes"] > 0
        assert probe["storage"]["probe_final_tq_materialized_bytes"] > 0
    numerical_probes = []
    samples = torch.randn(1024, HEAD_DIM, device=model.device, dtype=torch.float16)
    for bits in (2, 3, 4):
        quantizer = TurboQuantMSE(dim=HEAD_DIM, bits=bits, device=str(model.device))
        indices, norms = quantizer.quantize(samples); restored = quantizer.dequantize(indices, norms)
        nmse = float(((restored.float() - samples.float()).square().sum(-1) / samples.float().square().sum(-1).clamp_min(1e-12)).mean())
        assert indices.dtype == torch.uint8 and int(indices.max()) < 2 ** bits
        assert torch.isfinite(norms).all() and torch.isfinite(restored).all() and nmse <= TQ_NMSE_MAX_BY_BITS[bits]
        numerical_probes.append({"bits": bits, "normalized_mse": nmse, "physical_index_bytes_per_coordinate": indices.element_size()})
    validation = {"passed": True, "schema_version": SCHEMA_VERSION, "fingerprint": EXPERIMENT_FINGERPRINT,
                  "session_id": SESSION_ID, "runtime_sha256": RUNTIME_SHA256, "source_hashes": SOURCE_HASHES,
                  "fp16_architecture": fp16_gate, "awq_architecture_passed": True,
                  "parity_rows": parity_rows, "path_probes": serializable, "tq_numerical_probes": numerical_probes,
                  "validated_before_measurement": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
    validate_validation_record(validation)
    atomic_write_json(VALIDATION_JSON, validation)
    environment["status"] = "VALIDATION_PASSED_MEASUREMENTS_PENDING"
    environment["validation_file"] = VALIDATION_JSON.name
    atomic_write_json(ENVIRONMENT_JSON, environment)
finally:
    disable_variant(); release_model()
validate_validation_record(json.loads(VALIDATION_JSON.read_text()))
print("All architecture/parity/path gates passed")


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.69G [00:00<?, ?B/s]


INFO  LogBar: headless/AI-agent/notebook/CI environment detected; high-frequency progress bars and animations are disabled.
WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.
INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.
INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.          
INFO  

┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐
│ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │
└─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘
GPT-QModel   : 7.1.0
Transformers : 5.12.1
Torch        : 2.11.0+cu128
Triton       : 3.6.0
INFO  ExLlamaV2 AWQ: compiling torch.ops JIT e

Loading weights:   0%|          | 0/939 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

INFO  gc.collect() reclaimed 8348 objects in 0.377s                            
All architecture/parity/path gates passed


## 8. Fail-Closed 10 Hz Telemetry and Fixed-Work Energy Runner


In [8]:
def nvml_value(function, *args, default=np.nan):
    try: return function(*args)
    except (AttributeError, pynvml.NVMLError): return default

def sensor_sample():
    memory = pynvml.nvmlDeviceGetMemoryInfo(NVML_HANDLE)
    return {"power_w": float(pynvml.nvmlDeviceGetPowerUsage(NVML_HANDLE) / 1000),
            "nvml_used_mb": float(memory.used / 2**20),
            "temperature_c": float(nvml_value(pynvml.nvmlDeviceGetTemperature, NVML_HANDLE, pynvml.NVML_TEMPERATURE_GPU))}

def telemetry_snapshot():
    row = sensor_sample(); row.update({
        "sm_clock_mhz": float(nvml_value(pynvml.nvmlDeviceGetClockInfo, NVML_HANDLE, pynvml.NVML_CLOCK_SM)),
        "memory_clock_mhz": float(nvml_value(pynvml.nvmlDeviceGetClockInfo, NVML_HANDLE, pynvml.NVML_CLOCK_MEM)),
        "power_limit_w": float(pynvml.nvmlDeviceGetEnforcedPowerLimit(NVML_HANDLE) / 1000),
        "throttle_reasons": int(nvml_value(pynvml.nvmlDeviceGetCurrentClocksThrottleReasons, NVML_HANDLE, default=0)),
    }); assert np.isclose(row["power_limit_w"], 70.0, atol=0.5); return row

class GPUMonitor:
    def __init__(self, interval=POLL_INTERVAL_S):
        self.interval, self.samples, self.error, self.running = interval, [], None, False
        self.stop_event, self.lock = threading.Event(), threading.Lock()
    def sample(self, elapsed_ns=None):
        row = sensor_sample(); row["elapsed_s"] = (time.perf_counter_ns() - self.t0_ns) / 1e9 if elapsed_ns is None else elapsed_ns / 1e9
        with self.lock: self.samples.append(row)
    def poll(self):
        deadline = self.t0_ns + int(self.interval * 1e9)
        while not self.stop_event.wait(max(0, (deadline - time.perf_counter_ns()) / 1e9)):
            try: self.sample()
            except Exception as exc: self.error = repr(exc); self.stop_event.set(); return
            deadline += int(self.interval * 1e9)
    def start(self):
        assert not self.running; torch.cuda.synchronize(); self.samples = []; self.error = None; self.stop_event.clear()
        self.t0_ns = time.perf_counter_ns(); self.sample(0); self.running = True
        self.thread = threading.Thread(target=self.poll, daemon=True); self.thread.start()
    def stop(self):
        assert self.running; torch.cuda.synchronize(); self.stop_event.set(); self.thread.join(timeout=3)
        assert not self.thread.is_alive(); self.sample(); self.running = False
        if self.error: raise RuntimeError(f"NVML polling failed: {self.error}")
        rows = sorted(self.samples, key=lambda r: r["elapsed_s"]); assert len(rows) >= 2
        t = np.asarray([r["elapsed_s"] for r in rows]); p = np.asarray([r["power_w"] for r in rows])
        assert np.all(np.diff(t) >= 0) and np.isfinite(p).all() and (p >= 0).all() and t[-1] > 0
        if len(t) > 2:
            gaps = np.diff(t); assert np.median(gaps) <= 0.15 and gaps.max() <= 0.5
            assert len(t) >= max(2, int(t[-1] / self.interval) - 1)
        energies = {"raw": float(np.trapezoid(p, t)), "cap82": float(np.trapezoid(np.minimum(p, 82), t)),
                    "cap70": float(np.trapezoid(np.minimum(p, 70), t))}
        return {"duration_s": float(t[-1]), "energies": energies, "avg_power_w": energies["raw"] / t[-1],
                "max_power_w": float(p.max()), "n_samples": len(rows), "nvml_peak_mb": max(r["nvml_used_mb"] for r in rows),
                "temperature_max_c": max(r["temperature_c"] for r in rows), "samples": rows}

def measure_idle(label, seconds=10):
    samples, deadline = [], time.perf_counter() + seconds
    while True:
        samples.append(sensor_sample())
        if time.perf_counter() >= deadline: break
        time.sleep(POLL_INTERVAL_S)
    p = np.asarray([r["power_w"] for r in samples])
    return {"label": label, "samples": len(samples), "seconds": seconds, "power_median_w": float(np.median(p)),
            "power_mean_w": float(p.mean()), "power_sd_w": float(p.std(ddof=1))}

def gpu_memory_snapshot():
    torch.cuda.synchronize(); m = pynvml.nvmlDeviceGetMemoryInfo(NVML_HANDLE)
    return {"allocated_mb": torch.cuda.memory_allocated() / 2**20, "reserved_mb": torch.cuda.memory_reserved() / 2**20,
            "nvml_used_mb": m.used / 2**20}

def require_validation():
    return validate_validation_record(json.loads(VALIDATION_JSON.read_text()))

def run_energy_case(model, item, idle, stage, warmup=False):
    require_validation(); variant, context = item["variant"], item["context"]; spec = VARIANT_SPECS[variant]
    clear_adapter_logs();
    if spec["wrapper"]: enable_variant(variant, context)
    cache = new_cache(model, variant)
    ids = torch.tensor([ENERGY_PROMPT_IDS[context]], dtype=torch.long, device=model.device)
    prefill_monitor = decode_monitor = None
    try:
        clear_gpu(); before = telemetry_snapshot(); torch.cuda.reset_peak_memory_stats()
        prefill_monitor = GPUMonitor(); prefill_monitor.start(); start = time.perf_counter()
        with torch.inference_mode():
            model(input_ids=ids, position_ids=torch.arange(context, device=model.device).unsqueeze(0),
                  past_key_values=cache, use_cache=True, logits_to_keep=1)
        torch.cuda.synchronize(); prefill_s = time.perf_counter() - start; prefill = prefill_monitor.stop()
        prefill_lengths = cache_lengths(cache); prefill_memory = gpu_memory_snapshot(); prefill_storage = cache_storage_metrics(cache, "post_prefill")
        prefill_peak_alloc = torch.cuda.max_memory_allocated(); prefill_peak_reserved = torch.cuda.max_memory_reserved()
        torch.cuda.reset_peak_memory_stats(); decode_monitor = GPUMonitor(); decode_monitor.start(); start = time.perf_counter()
        with torch.inference_mode():
            for step, token_id in enumerate(TRACE_IDS):
                token = torch.tensor([[token_id]], device=model.device); position = torch.tensor([[context + step]], device=model.device)
                model(input_ids=token, position_ids=position, attention_mask={"full_attention": None},
                      past_key_values=cache, use_cache=True, logits_to_keep=1)
        torch.cuda.synchronize(); decode_s = time.perf_counter() - start; decode = decode_monitor.stop()
        structure = (validate_cache_path(cache, variant, context, prefill_lengths, 64) if spec["wrapper"]
                     else validate_stock_cache(cache, context, prefill_lengths, 64))
        decode_memory = gpu_memory_snapshot(); decode_storage = cache_storage_metrics(cache, "post_decode")
        after = telemetry_snapshot(); assert before["power_limit_w"] == after["power_limit_w"]
        energies = {r: prefill["energies"][r] + decode["energies"][r] for r in ("raw", "cap82", "cap70")}
        monitored = prefill["duration_s"] + decode["duration_s"]
        row = {"schema_version": SCHEMA_VERSION, "fingerprint": EXPERIMENT_FINGERPRINT, "session_id": SESSION_ID,
               "runtime_sha256": RUNTIME_SHA256,
               "block_attempt_id": stage.name, "schedule_sha256": SCHEDULE_SHA256, "semantic_sha256": EXPECTED_SEMANTIC_SHA256,
               "prompt_set_sha256": EXPECTED_PROMPT_SET_SHA256, "table": item["table"], "variant": variant,
               "context": context, "block": item["block"], "schedule_order": item["order"],
               "model_revision": MODEL_FP16_REVISION if variant == "Q0" else MODEL_AWQ_REVISION,
               "average_budget": spec["budget"], "tq_bits": spec["bits"],
               "energy_prompt_sha256": ENERGY_PROMPT_HASHES[str(context)], "decode_ids_sha256": TRACE_IDS_SHA256,
               "fixed_decode_tokens": 64, "prefill_s": prefill_s, "decode_s": decode_s,
               "prefill_energy_j": prefill["energies"]["raw"], "decode_energy_j": decode["energies"]["raw"],
               "total_energy_j": energies["raw"],
               "cap82_prefill_energy_j": prefill["energies"]["cap82"], "cap82_decode_energy_j": decode["energies"]["cap82"], "cap82_total_energy_j": energies["cap82"],
               "cap70_prefill_energy_j": prefill["energies"]["cap70"], "cap70_decode_energy_j": decode["energies"]["cap70"], "cap70_total_energy_j": energies["cap70"],
               "idle_adjusted_energy_j": energies["raw"] - idle["power_median_w"] * monitored,
               "decode_tok_s": 64 / decode_s, "avg_power_w": energies["raw"] / monitored,
               "max_power_w": max(prefill["max_power_w"], decode["max_power_w"]),
               "power_samples": prefill["n_samples"] + decode["n_samples"], "idle_power_w": idle["power_median_w"],
               "temperature_before_c": before["temperature_c"], "temperature_after_c": after["temperature_c"],
               "temperature_max_c": max(prefill["temperature_max_c"], decode["temperature_max_c"]),
               "sm_clock_before_mhz": before["sm_clock_mhz"], "sm_clock_after_mhz": after["sm_clock_mhz"],
               "memory_clock_before_mhz": before["memory_clock_mhz"], "memory_clock_after_mhz": after["memory_clock_mhz"],
               "power_limit_w": after["power_limit_w"], "throttle_before": before["throttle_reasons"], "throttle_after": after["throttle_reasons"],
               "prefill_peak_allocated_bytes": prefill_peak_alloc, "prefill_peak_reserved_bytes": prefill_peak_reserved,
               "decode_peak_allocated_bytes": torch.cuda.max_memory_allocated(), "decode_peak_reserved_bytes": torch.cuda.max_memory_reserved(),
               "post_prefill_nvml_used_mb": prefill_memory["nvml_used_mb"], "post_decode_nvml_used_mb": decode_memory["nvml_used_mb"],
               "nvml_sampled_peak_mb": max(prefill["nvml_peak_mb"], decode["nvml_peak_mb"]),
               **prefill_storage, **decode_storage, **structure}
        if warmup: return row
        stem = row_name(item["table"], variant, context, item["block"]).removesuffix(".json")
        trace = {"schema_version": SCHEMA_VERSION, "fingerprint": EXPERIMENT_FINGERPRINT, "table": item["table"],
                 "variant": variant, "context": context, "block": item["block"], "decode_ids_sha256": TRACE_IDS_SHA256,
                 "prefill": prefill["samples"], "decode": decode["samples"]}
        atomic_write_json(stage / f"{stem}__trace.json", trace)
        row["trace_name"] = f"{stem}__trace.json"; row["trace_sha256"] = hashlib.sha256((stage / row["trace_name"]).read_bytes()).hexdigest()
        atomic_write_json(stage / f"{stem}__row.json", row); return row
    finally:
        for monitor in (prefill_monitor, decode_monitor):
            if monitor is not None and monitor.running:
                try: monitor.stop()
                except Exception: pass
        if spec["wrapper"]: disable_variant()
        del cache, ids; clear_adapter_logs(); clear_gpu()


## 9. Paired Block Energy Collection

Q0 and AWQ are separate resident phases. Within each phase, a block is promoted only after every scheduled row and trace passes.


In [9]:
def run_energy_phase(phase, model, idle):
    for item in expected_block_items(phase, 0): run_energy_case(model, item, idle, DIRS["staging"], warmup=True)
    for block in range(MEASURED_BLOCKS):
        if recover_incomplete_block(phase, block): continue
        stale = list(DIRS["staging"].glob(f"{phase}_block{block}_*"))
        for old in stale:
            shutil.move(str(old), DIRS["failed_attempts"] / old.name)
        stage = DIRS["staging"] / f"{phase}_block{block}_{SESSION_ID}"; stage.mkdir()
        items = expected_block_items(phase, block)
        for item in items:
            row = run_energy_case(model, item, idle, stage, False)
            print("energy", item["table"], item["variant"], item["context"], block, round(row["total_energy_j"], 2))
        promote_energy_block(phase, block, stage, items)

require_validation(); model = load_fp16()
try:
    idle_q0 = measure_idle("Q0_FP16_model_resident"); atomic_write_json(SAVE_DIR / f"idle_q0_{SESSION_ID}.json", idle_q0)
    run_energy_phase("q0", model, idle_q0)
finally: release_model()
model = load_awq()
try:
    idle_awq = measure_idle("AWQ_model_resident"); atomic_write_json(SAVE_DIR / f"idle_awq_{SESSION_ID}.json", idle_awq)
    run_energy_phase("awq", model, idle_awq)
finally: disable_variant(); release_model()
core_energy_rows = refresh_csv(DIRS["core_energy_rows"], "core_energy_results.csv")
sensitivity_energy_rows = refresh_csv(DIRS["sensitivity_energy_rows"], "sensitivity_energy_results.csv")
assert len(core_energy_rows) == 75
assert len(sensitivity_energy_rows) == 20
assert len(list(DIRS["power_traces"].glob("*.json"))) == 95
print("Energy matrix complete: 95 rows and 95 traces")


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

energy core Q0 4096 0 606.1
energy core Q0 2048 0 342.53
energy core Q0 1024 0 245.15
energy core Q0 1024 1 237.24
energy core Q0 4096 1 621.05
energy core Q0 2048 1 314.13
energy core Q0 2048 2 356.11
energy core Q0 4096 2 616.17
energy core Q0 1024 2 280.17
energy core Q0 1024 3 235.03
energy core Q0 2048 3 339.52
energy core Q0 4096 3 616.58
energy core Q0 4096 4 623.97
energy core Q0 2048 4 318.48
energy core Q0 1024 4 295.28
INFO  Kernel: Auto-selection: adding candidate `AwqExllamaV2Linear`            
INFO  Kernel: selected -> `AwqExllamaV2Linear`.                                


Loading weights:   0%|          | 0/939 [00:00<?, ?it/s]

INFO  gc.collect() reclaimed 8131 objects in 0.390s                            
energy core Q1-W 4096 0 632.85
energy core Q2 4096 0 551.1
energy core Q3-Ref 1024 0 352.95
energy core Q1-W 2048 0 415.98
energy core Q4-Ref 4096 0 635.26
energy sensitivity Q2-B512 4096 0 503.17
energy core Q2 1024 0 262.77
energy core Q4-Ref 2048 0 417.83
energy sensitivity Q3-Ref-TQ2 4096 0 770.62
energy sensitivity Q3-Ref-TQ3 4096 0 773.82
energy sensitivity Q2-B1536 4096 0 599.68
energy core Q1-W 1024 0 255.87
energy core Q2 2048 0 339.74
energy core Q3-Ref 2048 0 543.54
energy core Q3-Ref 4096 0 814.64
energy core Q4-Ref 1024 0 350.23
energy sensitivity Q2-B1536 4096 1 577.57
energy core Q4-Ref 2048 1 443.23
energy core Q4-Ref 1024 1 336.12
energy sensitivity Q3-Ref-TQ3 4096 1 812.61
energy core Q2 4096 1 564.41
energy sensitivity Q2-B512 4096 1 521.48
energy core Q1-W 4096 1 598.98
energy core Q1-W 1024 1 300.75
energy core Q2 2048 1 339.4
energy sensitivity Q3-Ref-TQ2 4096 1 785.17
energy core Q2 1

## 10. Hard-Quality Runner and Teacher-Forced Diagnostics


In [10]:
def teacher_forced_metrics(model, variant, prompt):
    spec = VARIANT_SPECS[variant]; clear_adapter_logs()
    if spec["wrapper"]: enable_variant(variant, 4096)
    cache = new_cache(model, variant); ids = prompt_tensor(prompt, model)
    answer_ids = tokenizer.encode(prompt["target_code"], add_special_tokens=False); assert len(answer_ids) >= 2
    logps = []
    try:
        with torch.inference_mode():
            out = model(input_ids=ids, position_ids=torch.arange(4096, device=model.device).unsqueeze(0),
                        past_key_values=cache, use_cache=True, logits_to_keep=1)
            lengths = cache_lengths(cache); logits = out.logits[0, -1].float(); first = answer_ids[0]
            first_logp = float(F.log_softmax(logits, -1)[first]); rank = int((logits > logits[first]).sum() + 1)
            for i, current in enumerate(answer_ids[:-1]):
                token = torch.tensor([[current]], device=model.device); position = torch.tensor([[4096 + i]], device=model.device)
                out = model(input_ids=token, position_ids=position, attention_mask={"full_attention": None},
                            past_key_values=cache, use_cache=True, logits_to_keep=1)
                logps.append(float(F.log_softmax(out.logits[0, -1].float(), -1)[answer_ids[i + 1]]))
        structure = (validate_cache_path(cache, variant, 4096, lengths, len(answer_ids) - 1) if spec["wrapper"]
                     else validate_stock_cache(cache, 4096, lengths, len(answer_ids) - 1))
        return {"answer_token_ids": answer_ids, "answer_first_token_logprob": first_logp,
                "answer_first_token_nll": -first_logp, "answer_first_token_rank": rank,
                "answer_first_token_top1": rank == 1, "answer_nll_after_first_mean": float(-np.mean(logps)),
                "teacher_forced_structural_pass": structure["structural_pass"],
                "teacher_forced_active_cache_updates": structure.get("active_decode_cache_updates", 0),
                "teacher_forced_quantized_layers": structure.get("quantized_layers", 0),
                "teacher_forced_evicted_layers": structure.get("evicted_layers", 0),
                "teacher_forced_residual_only_layers": structure.get("residual_only_layers", []),
                "teacher_forced_tq_actual_path_max_nmse": structure.get("tq_actual_path_max_nmse", 0.0),
                "teacher_forced_stock_cache_validated": structure.get("stock_cache_validated", False)}
    finally:
        if spec["wrapper"]: disable_variant()
        del cache, ids; clear_adapter_logs(); clear_gpu()

def run_quality_case(model, item):
    variant, sid = item["variant"], item["scenario_id"]; prompt = prompts[sid]
    generation = run_generation(model, variant, prompt); scoring = score_response(generation["response"], prompt["target_code"])
    diagnostic = teacher_forced_metrics(model, variant, prompt); spec = VARIANT_SPECS[variant]
    return {"schema_version": SCHEMA_VERSION, "fingerprint": EXPERIMENT_FINGERPRINT, "session_id": SESSION_ID,
            "runtime_sha256": RUNTIME_SHA256,
            "schedule_sha256": SCHEDULE_SHA256, "semantic_sha256": EXPECTED_SEMANTIC_SHA256,
            "prompt_set_sha256": EXPECTED_PROMPT_SET_SHA256, "table": item["table"], "variant": variant,
            "scenario_id": sid, "schedule_order": item["order"], "model_revision": MODEL_FP16_REVISION if variant == "Q0" else MODEL_AWQ_REVISION,
            "depth": prompt["depth"], "context": 4096, "average_budget": spec["budget"], "tq_bits": spec["bits"],
            "target_key": prompt["target_key"], "target_code": prompt["target_code"], "input_ids_sha256": prompt["input_ids_sha256"],
            "response": generation["response"], "output_tokens": len(generation["output_ids"]), "output_token_ids": generation["output_ids"],
            "retrieval_correct": scoring["retrieval_correct"], "format_correct": scoring["format_correct"],
            "legacy_target_anywhere": scoring["legacy_target_anywhere"], "failure_category": scoring["failure_category"],
            "structural_pass": generation["structure"]["structural_pass"],
            "generation_active_decode_cache_updates": generation["structure"].get("active_decode_cache_updates", 0),
            "generation_quantized_layers": generation["structure"].get("quantized_layers", 0),
            "generation_evicted_layers": generation["structure"].get("evicted_layers", 0),
            "generation_residual_only_layers": generation["structure"].get("residual_only_layers", []),
            "generation_tq_actual_path_max_nmse": generation["structure"].get("tq_actual_path_max_nmse", 0.0),
            "generation_stock_cache_validated": generation["structure"].get("stock_cache_validated", False),
            **diagnostic}


## 11. Q0 and AWQ Hard-Quality Phases

Quality resumes per atomic row; core and fixed sensitivity-subset namespaces remain separate.


In [14]:
# Paste this entire file into one new Colab code cell. It patches the already
# loaded validator in memory, then resumes the quality schedules from saved rows.
# It is not intended to be run as a standalone experiment script.

def validate_structural_fields(row, prefix, decode_tokens):
    expected = expected_structure(row["variant"], row["context"], decode_tokens)
    key_prefix = f"{prefix}_" if prefix else ""
    active_key = (
        "teacher_forced_active_cache_updates"
        if prefix == "teacher_forced"
        else f"{key_prefix}active_decode_cache_updates"
    )
    assert row[active_key] == expected["active"]
    assert row[f"{key_prefix}evicted_layers"] == expected["evicted"]
    assert row[f"{key_prefix}quantized_layers"] == expected["quantized"]
    assert row[f"{key_prefix}residual_only_layers"] == expected["residual_only"]
    nmse = float(row[f"{key_prefix}tq_actual_path_max_nmse"])
    assert math.isfinite(nmse) and nmse >= 0
    if VARIANT_SPECS[row["variant"]]["tq"]:
        assert nmse <= TQ_NMSE_MAX_BY_BITS[row["tq_bits"]]
    else:
        assert nmse == 0


require_validation()
model = load_fp16()
try:
    done = {
        (row["variant"], row["scenario_id"])
        for row in read_shards(DIRS["core_quality_rows"])
    }
    for item in q0_quality_schedule:
        key = (item["variant"], item["scenario_id"])
        if key in done:
            continue
        row = run_quality_case(model, item)
        commit_quality_row(row, "core")
        done.add(key)
        print("quality", key, row["failure_category"])
finally:
    release_model()

model = load_awq()
try:
    core_done = {
        (row["variant"], row["scenario_id"])
        for row in read_shards(DIRS["core_quality_rows"])
    }
    sensitivity_done = {
        (row["variant"], row["scenario_id"])
        for row in read_shards(DIRS["sensitivity_quality_rows"])
    }
    for item in awq_quality_schedule:
        done = core_done if item["table"] == "core" else sensitivity_done
        key = (item["variant"], item["scenario_id"])
        if key in done:
            continue
        row = run_quality_case(model, item)
        commit_quality_row(row, item["table"])
        done.add(key)
        print("quality", item["table"], key, row["failure_category"])
finally:
    disable_variant()
    release_model()

refresh_csv(DIRS["core_quality_rows"], "core_quality_results.csv")
refresh_csv(DIRS["sensitivity_quality_rows"], "sensitivity_quality_results.csv")
assert len(read_shards(DIRS["core_quality_rows"])) == 300
assert len(read_shards(DIRS["sensitivity_quality_rows"])) == 60
print("Hard-quality matrix complete: 360 rows")


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

quality ('Q0', 'd10-s15') pass
quality ('Q0', 'd10-s12') pass
quality ('Q0', 'd10-s00') pass
quality ('Q0', 'd50-s14') wrong_code
quality ('Q0', 'd10-s02') pass
quality ('Q0', 'd10-s13') pass
quality ('Q0', 'd10-s11') pass
quality ('Q0', 'd50-s03') pass
quality ('Q0', 'd10-s09') pass
quality ('Q0', 'd50-s08') pass
quality ('Q0', 'd50-s10') pass
quality ('Q0', 'd50-s12') pass
quality ('Q0', 'd10-s04') wrong_code
quality ('Q0', 'd10-s06') pass
quality ('Q0', 'd90-s14') pass
quality ('Q0', 'd50-s17') pass
quality ('Q0', 'd10-s01') pass
quality ('Q0', 'd90-s04') pass
quality ('Q0', 'd50-s07') wrong_code
quality ('Q0', 'd90-s01') pass
quality ('Q0', 'd50-s15') pass
quality ('Q0', 'd50-s04') pass
quality ('Q0', 'd10-s16') pass
quality ('Q0', 'd10-s10') wrong_code
quality ('Q0', 'd50-s06') wrong_code
quality ('Q0', 'd10-s08') pass
quality ('Q0', 'd90-s17') pass
quality ('Q0', 'd10-s19') pass
quality ('Q0', 'd10-s07') pass
quality ('Q0', 'd90-s05') pass
quality ('Q0', 'd90-s06') pass
quality (

Loading weights:   0%|          | 0/939 [00:00<?, ?it/s]

INFO  gc.collect() reclaimed 8131 objects in 0.392s                            
quality core ('Q2', 'd10-s00') truncated_code
quality core ('Q1-W', 'd10-s00') pass
quality core ('Q4-Ref', 'd10-s00') truncated_code
quality core ('Q3-Ref', 'd10-s00') pass
quality sensitivity ('Q3-Ref-TQ3', 'd10-s00') pass
quality sensitivity ('Q3-Ref-TQ2', 'd10-s00') wrong_code
quality sensitivity ('Q2-B1536', 'd10-s00') truncated_code
quality sensitivity ('Q2-B512', 'd10-s00') no_code
quality core ('Q4-Ref', 'd10-s01') truncated_code
quality core ('Q1-W', 'd10-s01') pass
quality core ('Q2', 'd10-s01') truncated_code
quality core ('Q3-Ref', 'd10-s01') pass
quality sensitivity ('Q2-B512', 'd10-s01') truncated_code
quality sensitivity ('Q2-B1536', 'd10-s01') truncated_code
quality sensitivity ('Q3-Ref-TQ3', 'd10-s01') wrong_code
quality sensitivity ('Q3-Ref-TQ2', 'd10-s01') wrong_code
quality core ('Q2', 'd10-s02') wrong_code
quality core ('Q4-Ref', 'd10-s02') wrong_code
quality core ('Q1-W', 'd10-s02') pa

## 12. Bounded Quality Uncertainty and Registered Paired Comparisons


In [15]:
def read_bool(series):
    if series.dtype == bool: return series
    normalized = series.astype(str).str.casefold(); assert normalized.isin(["true", "false"]).all(); return normalized == "true"
def wilson(successes, n):
    z = stats.norm.ppf(.975); p = successes / n; d = 1 + z*z/n
    c = (p + z*z/(2*n))/d; h = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n))/d
    return max(0, c-h), min(1, c+h)
def paired_flips(frame, comparisons, family_name):
    rows = []
    for test_variant, baseline_variant in comparisons:
        b = frame[frame.variant == baseline_variant][["scenario_id", "depth", "retrieval_correct"]].rename(columns={"retrieval_correct":"baseline"})
        t = frame[frame.variant == test_variant][["scenario_id", "depth", "retrieval_correct"]].rename(columns={"depth":"test_depth", "retrieval_correct":"test"})
        pair = b.merge(t, on="scenario_id", validate="one_to_one"); assert (pair.depth == pair.test_depth).all()
        for depth, group in [("all", pair), *list(pair.groupby("depth"))]:
            n01 = int((~group.baseline & group.test).sum()); n10 = int((group.baseline & ~group.test).sum()); disc = n01+n10
            p = float(stats.binomtest(min(n01,n10), disc, .5).pvalue) if disc else 1.0
            rows.append({"family": family_name, "comparison": f"{test_variant}-{baseline_variant}", "depth": depth,
                         "n": len(group), "gain_n01": n01, "loss_n10": n10, "paired_difference": (n01-n10)/len(group), "p_exact": p})
    result = pd.DataFrame(rows); result["p_bonferroni"] = np.minimum(1, result.p_exact * len(result)); return result

core_quality = pd.DataFrame(read_shards(DIRS["core_quality_rows"])); sensitivity_quality = pd.DataFrame(read_shards(DIRS["sensitivity_quality_rows"]))
assert len(core_quality) == 300 and len(sensitivity_quality) == 60
for frame in (core_quality, sensitivity_quality):
    for col in ("retrieval_correct", "format_correct", "legacy_target_anywhere"): frame[col] = read_bool(frame[col])
    assert read_bool(frame.structural_pass).all() and read_bool(frame.teacher_forced_structural_pass).all()
interval_rows = []
for metric in ("retrieval_correct", "format_correct", "legacy_target_anywhere"):
    for keys, group in core_quality.groupby(["variant", "depth"], sort=True):
        s = int(group[metric].sum()); lo, hi = wilson(s, len(group)); interval_rows.append({"variant": keys[0], "depth": keys[1], "metric": metric, "successes": s, "n": len(group), "rate": s/len(group), "wilson95_low": lo, "wilson95_high": hi})
    for variant, group in core_quality.groupby("variant"):
        s = int(group[metric].sum()); lo, hi = wilson(s, len(group)); interval_rows.append({"variant": variant, "depth": "all", "metric": metric, "successes": s, "n": len(group), "rate": s/len(group), "wilson95_low": lo, "wilson95_high": hi})
pd.DataFrame(interval_rows).to_csv(DIRS["analysis"] / "quality_wilson_intervals.csv", index=False)
core_flips = paired_flips(core_quality, [(v,"Q1-W") for v in ("Q2","Q3-Ref","Q4-Ref")], "core_bonferroni_12")
core_flips.to_csv(DIRS["analysis"] / "core_paired_prediction_flips.csv", index=False); assert len(core_flips) == 12
subset_core = core_quality[core_quality.scenario_id.isin(sensitivity_scenario_ids)]
combined = pd.concat([subset_core, sensitivity_quality], ignore_index=True)
sensitivity_flips = paired_flips(combined, [("Q2-B512","Q2"),("Q2-B1536","Q2"),("Q3-Ref-TQ2","Q3-Ref"),("Q3-Ref-TQ3","Q3-Ref")], "sensitivity_bonferroni_16")
sensitivity_flips.to_csv(DIRS["analysis"] / "sensitivity_paired_prediction_flips.csv", index=False); assert len(sensitivity_flips) == 16
pd.concat([core_quality, sensitivity_quality], ignore_index=True).groupby(
    ["table","variant","depth","failure_category"]
).size().rename("count").reset_index().to_csv(DIRS["analysis"] / "failure_categories.csv", index=False)


## 13. Energy Statistics, Raw/Capped Robustness, and Uncertainty-Aware Pareto


In [17]:
# Paste this entire file into one new Colab code cell and run it.
# It reruns only derived analysis from saved rows/traces; no model is loaded.

core_energy = pd.DataFrame(read_shards(DIRS["core_energy_rows"])); sensitivity_energy = pd.DataFrame(read_shards(DIRS["sensitivity_energy_rows"])); energy = pd.concat([core_energy, sensitivity_energy])
assert (len(core_energy), len(sensitivity_energy)) == (75, 20)
trace_rows = []
for row in energy.to_dict("records"):
    path = DIRS["power_traces"] / row["trace_name"]; assert hashlib.sha256(path.read_bytes()).hexdigest() == row["trace_sha256"]
    trace = json.loads(path.read_text()); record = {"table": row["table"], "variant": row["variant"], "context": row["context"], "block": row["block"], "trace_name": path.name}
    values = validate_trace_and_row(row, path)
    for regime in ("raw", "cap82", "cap70"):
        record[f"{regime}_prefill_energy_j"] = values[f"{regime}_prefill"]
        record[f"{regime}_decode_energy_j"] = values[f"{regime}_decode"]
        record[f"{regime}_total_energy_j"] = values[f"{regime}_total"]
    trace_rows.append(record)
traces = pd.DataFrame(trace_rows); assert len(traces)==95; traces.to_csv(DIRS["analysis"] / "trace_level_power_robustness.csv", index=False)
summary=[]
for (table,variant,context),g in traces.groupby(["table","variant","context"]):
    r={"table":table,"variant":variant,"context":context,"n":len(g)}
    for regime in ("raw","cap82","cap70"):
        r[f"{regime}_mean_j"]=g[f"{regime}_total_energy_j"].mean(); r[f"{regime}_sd_j"]=g[f"{regime}_total_energy_j"].std(ddof=1)
    summary.append(r)
condition_summary=pd.DataFrame(summary); assert len(condition_summary)==19; condition_summary.to_csv(DIRS["analysis"] / "condition_power_summary.csv",index=False)

def finite_paired_t(a, b):
    difference = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    if np.allclose(difference, 0): return 0.0, 1.0
    if np.isclose(difference.std(ddof=1), 0): return math.copysign(float("inf"), difference.mean()), 0.0
    result = stats.ttest_rel(a, b); assert np.isfinite(result.statistic) and np.isfinite(result.pvalue)
    return float(result.statistic), float(result.pvalue)

def finite_onesample_t(values):
    values = np.asarray(values, dtype=float)
    if np.allclose(values, 0): return 0.0, 1.0
    if np.isclose(values.std(ddof=1), 0): return math.copysign(float("inf"), values.mean()), 0.0
    result = stats.ttest_1samp(values, 0); assert np.isfinite(result.statistic) and np.isfinite(result.pvalue)
    return float(result.statistic), float(result.pvalue)

def holm(pvalues):
    p=np.asarray(pvalues, dtype=float); assert np.isfinite(p).all() and ((0 <= p) & (p <= 1)).all()
    order=np.argsort(p); adjusted=np.empty(len(p)); running=0.0
    for rank,index in enumerate(order): running=max(running,(len(p)-rank)*p[index]); adjusted[index]=min(1.0,running)
    return adjusted
contrasts=[tuple(pair) for pair in ANALYSIS_PROTOCOL["energy_contrasts"]]
metric_map=ANALYSIS_PROTOCOL["energy_metrics"]
test_rows=[]
for regime in ("raw","cap82","cap70"):
    frame=core_energy[core_energy.context==4096].copy()
    if regime != "raw":
        frame["total_energy_j"] = frame[f"{regime}_total_energy_j"]
        frame["decode_energy_j"] = frame[f"{regime}_decode_energy_j"]
    start=len(test_rows)
    for left,right in contrasts:
        for label,column in metric_map.items():
            a=frame[frame.variant==left].sort_values("block")[column].to_numpy(); b=frame[frame.variant==right].sort_values("block")[column].to_numpy(); d=a-b
            t_value, p_value = finite_paired_t(a, b)
            test_rows.append({"regime":regime,"comparison":f"{left}-{right}","metric":label,"n":5,"mean_difference":d.mean(),
                              "t":t_value,"p_raw":p_value,
                              "wilcoxon_p":float(stats.wilcoxon(d).pvalue) if np.any(d) else 1.0})
    adjusted=holm([r["p_raw"] for r in test_rows[start:]])
    for r,p in zip(test_rows[start:],adjusted): r["p_holm_20"]=float(p); r["significant_0_05"]=bool(p<.05)
tests=pd.DataFrame(test_rows); assert len(tests)==60; tests.to_csv(DIRS["analysis"] / "core_energy_tests_by_regime.csv",index=False)

interaction_rows=[]
for regime in ("raw","cap82","cap70"):
    frame=core_energy[core_energy.context==4096].copy()
    if regime != "raw":
        frame["total_energy_j"] = frame[f"{regime}_total_energy_j"]
        frame["decode_energy_j"] = frame[f"{regime}_decode_energy_j"]
    start=len(interaction_rows)
    for metric,column in metric_map.items():
        pivot=frame.pivot(index="block",columns="variant",values=column).sort_index()
        values=(pivot["Q4-Ref"]-pivot["Q3-Ref"])-(pivot["Q2"]-pivot["Q1-W"])
        t_value, p_value = finite_onesample_t(values)
        interaction_rows.append({"regime":regime,"metric":metric,"n":5,"mean_interaction":values.mean(),
                                 "t":t_value,"p_raw":p_value})
    adjusted=holm([r["p_raw"] for r in interaction_rows[start:]])
    for row,p in zip(interaction_rows[start:],adjusted): row["p_holm_4"]=float(p)
interactions=pd.DataFrame(interaction_rows); assert len(interactions)==12
interactions.to_csv(DIRS["analysis"] / "interaction_tests_by_regime.csv",index=False)
sensitivity_summary=[]
for axis,baseline,variants in (("pyramid","Q2",["Q2-B512","Q2-B1536"]),("tq","Q3-Ref",["Q3-Ref-TQ2","Q3-Ref-TQ3"])):
    for variant in variants:
        left=sensitivity_energy[sensitivity_energy.variant==variant].sort_values("block")
        right=core_energy[(core_energy.variant==baseline)&(core_energy.context==4096)].sort_values("block")
        assert list(left.block)==list(right.block)==list(range(5))
        difference=left.total_energy_j.to_numpy()-right.total_energy_j.to_numpy()
        sensitivity_summary.append({"axis":axis,"variant":variant,"baseline":baseline,"n":5,
                                    "mean_energy_difference_j":difference.mean(),"sd_difference_j":difference.std(ddof=1)})
pd.DataFrame(sensitivity_summary).to_csv(DIRS["analysis"] / "sensitivity_energy_summary.csv",index=False)

raw=tests[tests.regime=="raw"][["comparison","metric","significant_0_05"]].rename(columns={"significant_0_05":"raw_significant"})
stability=[]
for regime in ("cap82","cap70"):
    cap=tests[tests.regime==regime][["comparison","metric","significant_0_05"]]
    joined=raw.merge(cap,on=["comparison","metric"])
    for _,r in joined.iterrows(): stability.append({"regime":regime,"comparison":r.comparison,"metric":r.metric,"raw_significant":r.raw_significant,"capped_significant":r.significant_0_05,"stable":r.raw_significant==r.significant_0_05})
significance=pd.DataFrame(stability); assert len(significance)==40; significance.to_csv(DIRS["analysis"] / "significance_stability.csv",index=False)

rankings=[]; rank_stability=[]
for context in CONTEXTS:
    orders={}
    for regime in ("raw","cap82","cap70"):
        group=condition_summary[(condition_summary.table=="core")&(condition_summary.context==context)].sort_values(f"{regime}_mean_j")
        orders[regime]=group.variant.tolist()
        for rank,(_,r) in enumerate(group.iterrows(),1): rankings.append({"axis":f"core_ctx{context}","regime":regime,"rank":rank,"variant":r.variant,"mean_j":r[f"{regime}_mean_j"]})
    rank_stability.append({"axis":f"core_ctx{context}","stable_82":orders["raw"]==orders["cap82"],"stable_70":orders["raw"]==orders["cap70"]})
for axis,levels in (("pyramid",["Q2-B512","Q2","Q2-B1536"]),("tq",["Q3-Ref-TQ2","Q3-Ref-TQ3","Q3-Ref"])):
    source=pd.concat([condition_summary[(condition_summary.table=="sensitivity")],condition_summary[(condition_summary.table=="core")&(condition_summary.context==4096)]])
    orders={}
    for regime in ("raw","cap82","cap70"):
        group=source[source.variant.isin(levels)].sort_values(f"{regime}_mean_j"); orders[regime]=group.variant.tolist()
        for rank,(_,r) in enumerate(group.iterrows(),1): rankings.append({"axis":axis,"regime":regime,"rank":rank,"variant":r.variant,"mean_j":r[f"{regime}_mean_j"]})
    rank_stability.append({"axis":axis,"stable_82":orders["raw"]==orders["cap82"],"stable_70":orders["raw"]==orders["cap70"]})
rankings=pd.DataFrame(rankings); assert len(rankings)==63; rankings.to_csv(DIRS["analysis"] / "energy_rankings_by_regime.csv",index=False)
rank_stability=pd.DataFrame(rank_stability); assert len(rank_stability)==5; rank_stability.to_csv(DIRS["analysis"] / "ranking_stability.csv",index=False)

def pareto_table(energy_frame, quality_frame, variants, repetitions=2000):
    points=[]
    for v in variants:
        e=energy_frame[(energy_frame.variant==v)&(energy_frame.context==4096)].sort_values("block").total_energy_j
        q=quality_frame[quality_frame.variant==v].sort_values("scenario_id").retrieval_correct.astype(float)
        assert len(e)==5 and len(q)>0
        lo,hi=wilson(int(q.sum()),len(q)); points.append({"variant":v,"energy_mean_j":e.mean(),"energy_sd_j":e.std(ddof=1),"retrieval_rate":q.mean(),"wilson95_low":lo,"wilson95_high":hi})
    result=pd.DataFrame(points); memberships={v:0 for v in variants}; rng=np.random.default_rng(SEED+99000)
    awq_variants=[v for v in variants if v!="Q0"]
    quality_ids_by_depth={depth:sorted(quality_frame[quality_frame.depth==depth].scenario_id.unique()) for depth in sorted(quality_frame.depth.unique())}
    for _ in range(repetitions):
        awq_blocks=rng.integers(0,5,5); q0_blocks=rng.integers(0,5,5)
        sampled_ids=[]
        for ids in quality_ids_by_depth.values(): sampled_ids.extend(np.asarray(ids)[rng.integers(0,len(ids),len(ids))].tolist())
        sampled=[]
        for v in variants:
            e=energy_frame[(energy_frame.variant==v)&(energy_frame.context==4096)].set_index("block").total_energy_j
            block_draw=q0_blocks if v=="Q0" else awq_blocks
            energy_mean=float(e.loc[block_draw].mean())
            q=quality_frame[quality_frame.variant==v].set_index("scenario_id").retrieval_correct.astype(float)
            quality_mean=float(q.loc[sampled_ids].mean())
            sampled.append((v,energy_mean,quality_mean))
        for v,e,q in sampled:
            dominated=any((oe<=e and oq>=q and (oe<e or oq>q)) for ov,oe,oq in sampled if ov!=v)
            memberships[v]+=not dominated
    result["frontier_membership_probability"]=result.variant.map(lambda v:memberships[v]/repetitions); return result
core_pareto=pareto_table(core_energy,core_quality,list(CORE_VARIANTS)); assert len(core_pareto)==5; core_pareto.to_csv(DIRS["analysis"] / "core_pareto_uncertainty.csv",index=False)
common_energy=pd.concat([core_energy,sensitivity_energy]); common_quality=pd.concat([subset_core,sensitivity_quality])
common_variants=list(CORE_VARIANTS)+list(SENSITIVITY_VARIANTS); common_pareto=pareto_table(common_energy,common_quality,common_variants); assert len(common_pareto)==9
common_pareto.to_csv(DIRS["analysis"] / "common15_pareto_uncertainty.csv",index=False)

analysis_core_energy_rows = read_shards(DIRS["core_energy_rows"])
analysis_sensitivity_energy_rows = read_shards(DIRS["sensitivity_energy_rows"])
analysis_core_quality_rows = read_shards(DIRS["core_quality_rows"])
analysis_sensitivity_quality_rows = read_shards(DIRS["sensitivity_quality_rows"])
analysis_input_sha256 = canonical_hash({
    "core_energy": sorted(analysis_core_energy_rows, key=lambda r: (r["variant"], r["context"], r["block"])),
    "sensitivity_energy": sorted(analysis_sensitivity_energy_rows, key=lambda r: (r["variant"], r["context"], r["block"])),
    "core_quality": sorted(analysis_core_quality_rows, key=lambda r: (r["variant"], r["scenario_id"])),
    "sensitivity_quality": sorted(analysis_sensitivity_quality_rows, key=lambda r: (r["variant"], r["scenario_id"])),
    "trace_hashes": sorted((row["trace_name"], row["trace_sha256"])
                           for row in analysis_core_energy_rows + analysis_sensitivity_energy_rows),
})
provenance_exclusions = {"analysis_provenance.json", "analysis_manifest.json", "final_integrity.json"}
output_hashes = {
    p.name: hashlib.sha256(p.read_bytes()).hexdigest()
    for p in sorted(DIRS["analysis"].iterdir()) if p.is_file() and p.name not in provenance_exclusions
}
atomic_write_json(DIRS["analysis"] / "analysis_provenance.json", {
    "fingerprint": EXPERIMENT_FINGERPRINT, "analysis_input_sha256": analysis_input_sha256,
    "output_sha256": output_hashes,
})


## 14. Final Integrity and Final Atomic Success Marker


In [18]:
validation_record = require_validation()
core_energy_rows=read_shards(DIRS["core_energy_rows"]); sensitivity_energy_rows=read_shards(DIRS["sensitivity_energy_rows"])
core_quality_rows=read_shards(DIRS["core_quality_rows"]); sensitivity_quality_rows=read_shards(DIRS["sensitivity_quality_rows"])
assert (len(core_energy_rows),len(sensitivity_energy_rows),len(core_quality_rows),len(sensitivity_quality_rows))==(75,20,300,60)
expected_core_energy=set((row["variant"],row["context"],row["block"]) for row in q0_energy_schedule+awq_energy_schedule if row["table"]=="core")
expected_sensitivity_energy=set((row["variant"],row["context"],row["block"]) for row in awq_energy_schedule if row["table"]=="sensitivity")
expected_core_quality=set((row["variant"],row["scenario_id"]) for row in q0_quality_schedule+awq_quality_schedule if row["table"]=="core")
expected_sensitivity_quality=set((row["variant"],row["scenario_id"]) for row in awq_quality_schedule if row["table"]=="sensitivity")
assert {(r["variant"],r["context"],r["block"]) for r in core_energy_rows}==expected_core_energy
assert {(r["variant"],r["context"],r["block"]) for r in sensitivity_energy_rows}==expected_sensitivity_energy
assert {(r["variant"],r["scenario_id"]) for r in core_quality_rows}==expected_core_quality
assert {(r["variant"],r["scenario_id"]) for r in sensitivity_quality_rows}==expected_sensitivity_quality
all_rows=core_energy_rows+sensitivity_energy_rows+core_quality_rows+sensitivity_quality_rows
assert len(all_rows)==455 and all(r["variant"]!="Q1-S" and r["structural_pass"] for r in all_rows)
traces=list(DIRS["power_traces"].glob("*.json")); assert len(traces)==95

core_energy_csv = pd.read_csv(SAVE_DIR / "core_energy_results.csv")
sensitivity_energy_csv = pd.read_csv(SAVE_DIR / "sensitivity_energy_results.csv")
assert len(core_energy_csv) == 75 and len(sensitivity_energy_csv) == 20
assert set(map(tuple, core_energy_csv[["table","variant","context","block"]].to_records(index=False))) == {
    (row["table"], row["variant"], row["context"], row["block"]) for row in core_energy_rows}
assert set(map(tuple, sensitivity_energy_csv[["table","variant","context","block"]].to_records(index=False))) == {
    (row["table"], row["variant"], row["context"], row["block"]) for row in sensitivity_energy_rows}

csv_snapshots = {
    "core_energy_results.csv": core_energy_rows,
    "sensitivity_energy_results.csv": sensitivity_energy_rows,
    "core_quality_results.csv": core_quality_rows,
    "sensitivity_quality_results.csv": sensitivity_quality_rows,
}
for csv_name, shard_rows in csv_snapshots.items():
    expected_csv = pd.DataFrame(shard_rows).to_csv(index=False)
    assert (SAVE_DIR / csv_name).read_text(encoding="utf-8") == expected_csv
all_energy_rows=core_energy_rows+sensitivity_energy_rows
assert len({row["trace_name"] for row in all_energy_rows})==95
assert {path.name for path in traces}=={row["trace_name"] for row in all_energy_rows}
for row in all_energy_rows: validate_trace_and_row(row, DIRS["power_traces"] / row["trace_name"])
expected_markers={f"{phase}_block{block}.json" for phase in ("q0","awq") for block in range(5)}
assert {p.name for p in DIRS["block_commits"].glob("*.json")}==expected_markers
for phase in ("q0","awq"):
    for block in range(5): assert recover_incomplete_block(phase,block)
assert len(pd.read_csv(DIRS["analysis"] / "trace_level_power_robustness.csv"))==95
assert len(pd.read_csv(DIRS["analysis"] / "condition_power_summary.csv"))==19
assert len(pd.read_csv(DIRS["analysis"] / "energy_rankings_by_regime.csv"))==63
assert len(pd.read_csv(DIRS["analysis"] / "ranking_stability.csv"))==5
assert len(pd.read_csv(DIRS["analysis"] / "core_energy_tests_by_regime.csv"))==60
assert len(pd.read_csv(DIRS["analysis"] / "significance_stability.csv"))==40
assert len(pd.read_csv(DIRS["analysis"] / "core_pareto_uncertainty.csv"))==5
assert len(pd.read_csv(DIRS["analysis"] / "common15_pareto_uncertainty.csv"))==9

recomputed_analysis_input_sha256 = canonical_hash({
    "core_energy": sorted(core_energy_rows, key=lambda r: (r["variant"], r["context"], r["block"])),
    "sensitivity_energy": sorted(sensitivity_energy_rows, key=lambda r: (r["variant"], r["context"], r["block"])),
    "core_quality": sorted(core_quality_rows, key=lambda r: (r["variant"], r["scenario_id"])),
    "sensitivity_quality": sorted(sensitivity_quality_rows, key=lambda r: (r["variant"], r["scenario_id"])),
    "trace_hashes": sorted((row["trace_name"], row["trace_sha256"]) for row in all_energy_rows),
})
provenance = json.loads((DIRS["analysis"] / "analysis_provenance.json").read_text())
assert provenance["fingerprint"] == EXPERIMENT_FINGERPRINT
assert provenance["analysis_input_sha256"] == recomputed_analysis_input_sha256
for name, expected_hash in provenance["output_sha256"].items():
    assert hashlib.sha256((DIRS["analysis"] / name).read_bytes()).hexdigest() == expected_hash
assert len(pd.read_csv(DIRS["analysis"] / "core_paired_prediction_flips.csv"))==12
assert len(pd.read_csv(DIRS["analysis"] / "sensitivity_paired_prediction_flips.csv"))==16
assert len(pd.read_csv(DIRS["analysis"] / "interaction_tests_by_regime.csv"))==12
assert len(pd.read_csv(DIRS["analysis"] / "sensitivity_energy_summary.csv"))==4
required_analysis={"quality_wilson_intervals.csv","core_paired_prediction_flips.csv","sensitivity_paired_prediction_flips.csv",
  "failure_categories.csv","trace_level_power_robustness.csv","condition_power_summary.csv","core_energy_tests_by_regime.csv",
  "significance_stability.csv","energy_rankings_by_regime.csv","ranking_stability.csv","core_pareto_uncertainty.csv","common15_pareto_uncertainty.csv",
  "interaction_tests_by_regime.csv","sensitivity_energy_summary.csv"}
assert required_analysis <= {p.name for p in DIRS["analysis"].iterdir() if p.is_file()}
analysis_manifest=[]
for p in sorted(DIRS["analysis"].iterdir()):
    if p.is_file() and p.name not in {"final_integrity.json","analysis_manifest.json"}: analysis_manifest.append({"path":p.name,"bytes":p.stat().st_size,"sha256":hashlib.sha256(p.read_bytes()).hexdigest()})
atomic_write_json(DIRS["analysis"] / "analysis_manifest.json", {"fingerprint":EXPERIMENT_FINGERPRINT,"files":analysis_manifest})

assert not any(path.is_file() for path in DIRS["staging"].rglob("*"))
artifact_manifest_path = SAVE_DIR / "artifact_manifest.json"
excluded_artifacts = {FINAL_JSON.resolve(), artifact_manifest_path.resolve()}
artifact_files = []
for artifact_path in sorted(SAVE_DIR.rglob("*")):
    if artifact_path.is_file() and artifact_path.resolve() not in excluded_artifacts:
        artifact_files.append({"path": artifact_path.relative_to(SAVE_DIR).as_posix(), "bytes": artifact_path.stat().st_size,
                               "sha256": hashlib.sha256(artifact_path.read_bytes()).hexdigest()})
assert len(artifact_files) == len({item["path"] for item in artifact_files})
atomic_write_json(artifact_manifest_path, {"fingerprint": EXPERIMENT_FINGERPRINT, "runtime_sha256": RUNTIME_SHA256,
                                          "validation_sha256": hashlib.sha256(VALIDATION_JSON.read_bytes()).hexdigest(),
                                          "files": artifact_files})
artifact_manifest_sha256 = hashlib.sha256(artifact_manifest_path.read_bytes()).hexdigest()
final={"passed":True,"schema_version":SCHEMA_VERSION,"fingerprint":EXPERIMENT_FINGERPRINT,"session_id":SESSION_ID,
       "runtime_sha256":RUNTIME_SHA256,"analysis_protocol_sha256":ANALYSIS_PROTOCOL_SHA256,
       "validation_sha256":hashlib.sha256(VALIDATION_JSON.read_bytes()).hexdigest(),
       "artifact_manifest":"artifact_manifest.json","artifact_manifest_sha256":artifact_manifest_sha256,
       "core_energy_rows":75,"sensitivity_energy_rows":20,"power_traces":95,"core_quality_rows":300,
       "sensitivity_quality_rows":60,"unique_measurement_keys":455,"q1s_measurement_rows":0,
       "semantic_sha256":EXPECTED_SEMANTIC_SHA256,"prompt_set_sha256":EXPECTED_PROMPT_SET_SHA256,
       "schedule_sha256":SCHEDULE_SHA256,"decode_ids_sha256":TRACE_IDS_SHA256,
       "trace_reintegration_passed":True,"raw_primary":True,"capped_sensitivity_regimes":[82,70],
       "core_pareto_rows":5,"common15_pareto_rows":9,"analysis_manifest":"analysis_manifest.json",
       "completed_at_utc":time.strftime("%Y-%m-%dT%H:%M:%SZ",time.gmtime())}
atomic_write_json(FINAL_JSON,final)  # Final filesystem mutation: do not write after this line.
print(json.dumps(final,indent=2))


{
  "passed": true,
  "schema_version": "track-a-qwen-full-reference-v1-2026-08-04-1",
  "fingerprint": "861d89617e53034d",
  "session_id": "20260804T154752-97822cdb",
  "runtime_sha256": "8883c3ef97ce51e55655e99fe2de97df51162b4de3574e07405bb4ca66dd2d71",
  "analysis_protocol_sha256": "f9fc97e0388874b150c413729bb74e3b1c8faaf7db421307aa7de9ff057cc9a0",
  "validation_sha256": "f05ef59b3eb3679b7de9a78a1f22f96c37430e50afad377612cdbd2c0b3b94be",
  "artifact_manifest": "artifact_manifest.json",
  "artifact_manifest_sha256": "975cb8f19405f9d7e4267bc4a8143ead943c853776c6ba88b8697695ac78c1c9",
  "core_energy_rows": 75,
  "sensitivity_energy_rows": 20,
  "power_traces": 95,
  "core_quality_rows": 300,
  "sensitivity_quality_rows": 60,
  "unique_measurement_keys": 455,
  "q1s_measurement_rows": 0,
  "semantic_sha256": "fffa82607af6e7d2c892d18d5bc790524570833472ad397b7ab2e54eb03005ef",
  "prompt_set_sha256": "115c1568fea7aeea038a46ce309d0c8bcaf1f5405544466ad6f379ee7e94ae3a",
  "schedule_sha256": "

## Acceptance Gate

Accept the run only when `analysis/final_integrity.json` reports `passed: true`, 455 unique measurement rows, 95 traces, complete raw/capped analyses, and both Pareto tables. After success, treat the entire Qwen reference directory as immutable and build the CPU-only cross-model synthesis notebook.
